# SEA-AD scDRS-FM analysis

Cleaned notebook for SEA-AD Alzheimer’s disease analyses. The notebook keeps the main scDRS-FM heatmaps, saves phenotype-score UMAPs and all-trait scDRS-FM/scDRS/scPagwas UMAPs, exports independent signal assignments, and reproduces the final pathway heatmap workflow.

For the Nature Genetics manuscript supplementary materials, every cell-type heatmap writes a tidy CSV in its exact plotted row/column order. SEA-AD `indep_cells` exports are restricted to subclass sections whose marginal × conditional proportion is **strictly greater than the main heatmap threshold (1%)**; exact-threshold values are excluded and placeholder signal labels such as `-1` are retained in passing sections.

Key plotting conventions:
- all heatmap cells are square,
- cell-type labels include counts,
- all cell types with at least one marginal association are retained,
- independent population annotations are white with a black outline,
- figure text uses **scDRS-FM** for the functional-mapping method.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Mapping, Optional, Sequence
import pickle
import re
import textwrap
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

try:
    from adjustText import adjust_text
except Exception:  # pragma: no cover - optional plotting nicety
    def adjust_text(texts, *args, **kwargs):
        return texts

sns.set_style("white")
warnings.filterwarnings("ignore", category=FutureWarning)

## Configuration

All paths are defined here. `first_existing_path` lets the notebook work whether paths are relative to the project root or to the original notebook location.

In [2]:
# === scDRS-FM reproduction: portable path anchor (injected, P5) ===
import os as _os
from pathlib import Path as _Path
def _find_repo_root():
    # 1) explicit override wins
    env = _os.environ.get('SCDRSFM_BASE')
    if env:
        return _Path(env)
    # 2) search upward from CWD for the reproduction repo root
    #    (a directory containing both 'scDRS-FM-main' and 'scripts')
    here = _Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / 'scDRS-FM-main').is_dir() and (d / 'scripts').is_dir():
            return d
    # 3) last resort: current working directory
    return here
BASE = _find_repo_root()
DATA = BASE / 'data'
RESULTS = BASE / 'results'
MAGMA_REF = BASE / 'magma_ref'
assert BASE.exists(), f'Repro root not found (set SCDRSFM_BASE to the repo root): {BASE}'


In [3]:
OUTPUT_DIR = Path("sea_ad_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INDEP_CELLS_DIR = Path("indep_cells/sea_ad")
INDEP_CELLS_DIR.mkdir(parents=True, exist_ok=True)

# Nature Genetics manuscript supplementary tables generated by cell-type heatmaps.
MANUSCRIPT_SUPPLEMENTARY_DIR = Path("nature_genetics_manuscript_supplementary")
MANUSCRIPT_SUPPLEMENTARY_DIR.mkdir(parents=True, exist_ok=True)

SEA_AD_SCDRSFM_SUBCLASS_CSV = MANUSCRIPT_SUPPLEMENTARY_DIR / "SEA_AD_scDRSFM_subclass_heatmap_cell_type_proportions.csv"
SEA_AD_SCDRSFM_REGION_STAR_CSV = MANUSCRIPT_SUPPLEMENTARY_DIR / "SEA_AD_scDRSFM_subclass_heatmap_region_supertype_stars_cell_type_proportions.csv"
SEA_AD_SCDRS_CSV = MANUSCRIPT_SUPPLEMENTARY_DIR / "SEA_AD_scDRS_all_traits_heatmap_cell_type_proportions.csv"
SEA_AD_SCPAGWAS_CSV = MANUSCRIPT_SUPPLEMENTARY_DIR / "SEA_AD_scPagwas_all_traits_heatmap_cell_type_proportions.csv"
BRAUN_SCDRSFM_CELLCLASS_CSV = MANUSCRIPT_SUPPLEMENTARY_DIR / "Braun_scDRSFM_cellclass_heatmap_cell_type_proportions.csv"
BRAUN_SCDRSFM_REGION_STAR_CSV = MANUSCRIPT_SUPPLEMENTARY_DIR / "Braun_scDRSFM_cellclass_heatmap_region_stars_cell_type_proportions.csv"


def first_existing_path(*candidates: str | Path) -> Path:
    """Return the first existing candidate, or the first candidate if none exists yet."""
    paths = [Path(p) for p in candidates]
    for path in paths:
        if path.exists():
            return path
    return paths[0]


SEA_AD_H5AD = first_existing_path(
    DATA / "subsets_10k" / "SEA_AD" / "combined_healthy_filtered.h5ad",
)
SEA_AD_SCDRSFM_DIR = first_existing_path(
    RESULTS / "ct" / "sea_ad_brain_magic_ctrl",
)
SEA_AD_SCDRS_DIR = RESULTS / "ct" / "sea_ad_brain_none_ctrl"
SEA_AD_SCPAGWAS_DIR = RESULTS / "scpagwas" / "sea_ad"  # absent -> guarded
SEA_AD_SCPAGWAS_AD_DIR = SEA_AD_SCPAGWAS_DIR / "PASS_Alzheimers_Jansen2019"
SEA_AD_PHENOTYPE_SCORE_DIR = first_existing_path(
    RESULTS / "real" / "sea_ad_micro",
)
SEA_AD_HM_SCORE_FILE = SEA_AD_PHENOTYPE_SCORE_DIR / "HM_gs.marginal_score.gz"

FUNCTIONAL_PHENOTYPES = ["HM_gs", "DAM_gs", "CRM_gs", "IRM_gs", "HLA_gs"]
PHENOTYPE_LABELS = {
    "HM_gs": "HM (-)",
    "DAM_gs": "DAM",
    "CRM_gs": "CRM",
    "IRM_gs": "IRM",
    "HLA_gs": "HLA",
}
PHENOTYPE_SCORE_MULTIPLIERS = {
    "HM_gs": -1.0,
    "DAM_gs": 1.0,
    "CRM_gs": 1.0,
    "IRM_gs": 1.0,
    "HLA_gs": 1.0,
}

BRAUN_H5AD = DATA / "subsets_10k" / "Braun" / "human_dev_layers_100k.h5ad"
BRAUN_SCDRSFM_DIR = RESULTS / "real" / "braun_micro"

AD_TRAIT = "PASS_Alzheimers_Jansen2019"
SUBSET_TRAITS = [
    "PASS_Parkinsons23andMe_Corces2020",
    "PASS_Alzheimers_Jansen2019",
    "PASS_ADHD_Demontis2018",
    "PASS_BIP_Mullins2021",
    "PASS_Intelligence_SavageJansen2018",
    "PASS_Schizophrenia_Pardinas2018",
    "PASS_MDD_Howard2019",
    "UKB_460K.mental_NEUROTICISM",
]
# Keep the complete scDRS-FM trait list even though a later cell narrows
# SUBSET_TRAITS to the traits available for scDRS/scPagwas comparisons.
SCDRSFM_TRAITS = list(SUBSET_TRAITS)

TRAIT_LABELS = {
    # Neurodegenerative
    "PASS_Parkinsons23andMe_Corces2020": "Parkinson’s Disease (PD)",
    "PASS_Alzheimers_Jansen2019": "Alzheimer’s Disease (AD)",

    # Neurodevelopmental / psychiatric
    "PASS_ADHD_Demontis2018": "ADHD",
    "PASS_BIP_Mullins2021": "Bipolar Disorder (BIP)",
    "PASS_Schizophrenia_Pardinas2018": "Schizophrenia (SCZ)",
    "PASS_MDD_Howard2019": "Major Depressive Disorder (MDD)",

    # Cognitive / personality
    "PASS_Intelligence_SavageJansen2018": "Intelligence (IQ)",
    "UKB_460K.mental_NEUROTICISM": "Neuroticism",

}

SEA_SUBCLASS_GROUPS = {
    "Glial": ["Microglia-PVM", "Astrocyte", "OPC", "Endothelial", "VLMC"],
    "Excitatory": ["L2/3 IT", "L4 IT", "L5 IT", "L5 ET", "L6 IT", "L6 IT Car3", "L6b", "L6 CT"],
    "Inhibitory": ["Vip", "Pvalb", "Sst", "Sst Chodl", "Lamp5","Lamp5 Lhx6", "Sncg", "Pax6", "Chandelier"],
}
SEA_GROUP_COLORS = {"Glial": "green", "Excitatory": "red", "Inhibitory": "blue", "Other": "black"}
SEA_SUBCLASS_ORDER = [ct for group in SEA_SUBCLASS_GROUPS.values() for ct in group]

HEATMAP_THRESHOLD = 0.01
REGION_SUPERTYPE_STAR_THRESHOLD = 0.05
FDR_ALPHA = 0.10

print("SEA-AD h5ad:", SEA_AD_H5AD)
print("SEA-AD scDRS-FM results:", SEA_AD_SCDRSFM_DIR)
print("SEA-AD scDRS results:", SEA_AD_SCDRS_DIR)
print("SEA-AD scPagwas AD results:", SEA_AD_SCPAGWAS_AD_DIR)
print("SEA-AD phenotype scores:", SEA_AD_PHENOTYPE_SCORE_DIR)


SEA-AD h5ad: /mnt/shared-workspace/scdrsfm/data/subsets_10k/SEA_AD/combined_healthy_filtered.h5ad
SEA-AD scDRS-FM results: /mnt/shared-workspace/scdrsfm/results/ct/sea_ad_brain_magic_ctrl
SEA-AD scDRS results: /mnt/shared-workspace/scdrsfm/results/ct/sea_ad_brain_none_ctrl
SEA-AD scPagwas AD results: /mnt/shared-workspace/scdrsfm/results/scpagwas/sea_ad/PASS_Alzheimers_Jansen2019
SEA-AD phenotype scores: /mnt/shared-workspace/scdrsfm/results/real/sea_ad_micro


## General helper functions

In [4]:
def pick_first_existing_col(df: pd.DataFrame, candidates: Sequence[str], *, what: str) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"{what}: none of these columns exist: {tuple(candidates)}")


def bh_fdr_mask(pvals: np.ndarray, alpha: float) -> np.ndarray:
    """Benjamini-Hochberg FDR reject mask, robust to NaNs."""
    p = np.asarray(pvals, dtype=float)
    ok = np.isfinite(p)
    out = np.zeros(len(p), dtype=bool)
    if ok.sum() == 0:
        return out
    reject, _, _, _ = multipletests(p[ok], alpha=alpha, method="fdr_bh")
    out[ok] = reject
    return out


def parse_csv_index(value: object) -> pd.Index:
    if value is None or pd.isna(value):
        return pd.Index([], dtype=str)
    text = str(value)
    if text == "" or text.lower() == "nan":
        return pd.Index([], dtype=str)
    return pd.Index([x for x in text.split(",") if x], dtype=str)


def join_index(index: Iterable[object]) -> str:
    idx = pd.Index(index).astype(str).unique()
    try:
        idx = idx.sort_values()
    except Exception:
        pass
    return ",".join(idx.tolist())


def capfirst(text: object) -> str:
    text = str(text).strip()
    return text[:1].upper() + text[1:] if text else text


def celltype_counts(adata: sc.AnnData, biocol: str) -> pd.Series:
    if biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs missing {biocol!r}")
    return adata.obs[biocol].astype(str).value_counts()


def labels_with_counts(cell_types: Sequence[str], counts: pd.Series) -> list[str]:
    return [f"{capfirst(ct)} ({int(counts.get(ct, 0)):,})" for ct in cell_types]


def create_square_heatmap_figure(
    n_rows: int,
    n_cols: int,
    *,
    cell_size: float = 0.62,
    left_margin: float = 6.8,
    right_margin: float = 3.2,
    bottom_margin: float = 6.8,
    top_margin: float = 4.2,
):
    """Create an axis sized so a one-unit heatmap tile is physically square."""
    heatmap_width = max(3.5, n_cols * cell_size)
    heatmap_height = max(3.5, n_rows * cell_size)
    fig_width = left_margin + heatmap_width + right_margin
    fig_height = bottom_margin + heatmap_height + top_margin
    fig = plt.figure(figsize=(fig_width, fig_height))
    ax_left = left_margin / fig_width
    ax_bottom = bottom_margin / fig_height
    ax_width = heatmap_width / fig_width
    ax_height = heatmap_height / fig_height
    ax = fig.add_axes([ax_left, ax_bottom, ax_width, ax_height])
    ax.set_aspect("equal", adjustable="box")
    return fig, ax, (ax_left, ax_bottom, ax_width, ax_height)


def add_top_group_lines(ax, color_counts, names, *, y_axes=1.01, text_offset=0.02, linewidth=4, fontsize=24):
    total = max(1, sum(c for _, c in color_counts))
    start = 0
    for (color, count), name in zip(color_counts, names):
        end = start + int(count)
        xmin, xmax = start / total, end / total
        if count > 0:
            ax.add_line(Line2D([xmin, xmax], [y_axes, y_axes], transform=ax.transAxes,
                               color=color, linewidth=linewidth, solid_capstyle="butt", clip_on=False))
            ax.text((xmin + xmax) / 2, y_axes + text_offset, name,
                    ha="center", va="bottom", fontsize=fontsize, color=color,
                    fontweight="bold", transform=ax.transAxes, clip_on=False)
        start = end


def color_ticklabels(ticklabels, colors, *, fontsize=20, rotation=None, ha=None):
    for tick, color in zip(ticklabels, colors):
        tick.set_color(color)
        tick.set_fontsize(fontsize)
        if rotation is not None:
            tick.set_rotation(rotation)
        if ha is not None:
            tick.set_ha(ha)


def ordered_by_reference(values: Sequence[str], reference: Sequence[str]) -> list[str]:
    values = [str(v) for v in values]
    present = set(values)
    ordered = [v for v in reference if v in present]
    ordered.extend(sorted(v for v in values if v not in set(ordered)))
    return ordered


def load_and_preprocess_adata(path: Path, *, min_genes=250, min_cells=50, normalize=False) -> sc.AnnData:
    adata = sc.read_h5ad(path)
    adata.obs_names = adata.obs_names.astype(str)
    print(f"Loaded {path}: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)
    if normalize:
        sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
        sc.pp.log1p(adata)
    print(f"After filtering: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
    return adata


def ensure_sea_ad_obs_columns(adata: sc.AnnData) -> None:
    """Create the label columns used throughout the SEA-AD analysis."""
    required = ["Brain Region", "Supertype", "Subclass"]
    missing = [c for c in required if c not in adata.obs.columns]
    if missing:
        raise ValueError(f"SEA-AD AnnData is missing required obs columns: {missing}")

    adata.obs["Brain Region"] = adata.obs["Brain Region"].astype(str)
    adata.obs["Supertype"] = adata.obs["Supertype"].astype(str)
    adata.obs["Subclass"] = adata.obs["Subclass"].astype(str)
    adata.obs["Region_Supertype"] = adata.obs["Brain Region"] + "_" + adata.obs["Supertype"]
    adata.obs["Region_Subclass"] = adata.obs["Brain Region"] + "_" + adata.obs["Subclass"]

## scDRS-FM result parsing and independent-cell exports

The table builder writes one gzip-compressed TSV per trait to `indep_cells/sea_ad/{trait}.gz`, with columns `marginal_x_conditional_cell_id` and `independent_signal`.

In [5]:
def cells_from_metacell_rows(df: pd.DataFrame, *, cell_ids_col: str = "cell_ids") -> pd.Index:
    if cell_ids_col not in df.columns:
        raise ValueError(f"Conditional file missing {cell_ids_col!r}")
    cells = pd.Index([], dtype=str)
    for value in df[cell_ids_col].astype(str).tolist():
        cells = cells.append(parse_csv_index(value))
    return pd.Index(cells.astype(str)).unique()


def discovery_fraction_by_celltype(
    adata: sc.AnnData,
    discovered_cell_ids: pd.Index,
    *,
    biocol: str,
    celltype_order: Sequence[str],
    totals_by_type: pd.Series,
) -> pd.Series:
    discovered_cell_ids = adata.obs_names.intersection(pd.Index(discovered_cell_ids).astype(str))
    if len(discovered_cell_ids) == 0:
        return pd.Series(0.0, index=celltype_order)
    counts = adata.obs.loc[discovered_cell_ids, biocol].astype(str).value_counts()
    frac = (counts / totals_by_type).reindex(celltype_order, fill_value=0.0)
    return frac.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)


def valid_signal_values(series: pd.Series) -> list[int]:
    values = pd.to_numeric(series, errors="coerce")
    values = values[values.notna() & (values >= 0)]
    return sorted(values.astype(int).unique().tolist())




def _passes_heatmap_inclusion(value: object, threshold: float) -> bool:
    """Return True only for a finite proportion strictly above the heatmap cutoff."""
    try:
        value_float = float(value)
        threshold_float = float(threshold)
    except (TypeError, ValueError):
        return False
    return bool(np.isfinite(value_float) and value_float > threshold_float)

def build_marginal_x_conditional_signal_assignments(
    *,
    adata: sc.AnnData,
    marginal_sig_cells: pd.Index,
    df_cond_sig: pd.DataFrame,
    indep_sig_col: str,
) -> pd.DataFrame:
    """Per-trait independent-signal export table."""
    columns = ["marginal_x_conditional_cell_id", "independent_signal"]
    if len(df_cond_sig) == 0:
        return pd.DataFrame(columns=columns)

    signal_values = pd.to_numeric(df_cond_sig[indep_sig_col], errors="coerce")
    signal_values = signal_values[signal_values.notna()]
    rows = []

    for signal_value in sorted(signal_values.unique()):
        sub = df_cond_sig.loc[pd.to_numeric(df_cond_sig[indep_sig_col], errors="coerce").eq(signal_value)]
        signal_cells = adata.obs_names.intersection(cells_from_metacell_rows(sub))
        cells = marginal_sig_cells.intersection(signal_cells)
        if len(cells) == 0:
            continue
        cells = pd.Index(cells.astype(str)).unique()
        try:
            cells = cells.sort_values()
        except Exception:
            pass
        rows.append(pd.DataFrame({
            "marginal_x_conditional_cell_id": cells,
            "independent_signal": int(signal_value) if float(signal_value).is_integer() else signal_value,
        }))

    if not rows:
        return pd.DataFrame(columns=columns)
    return pd.concat(rows, axis=0, ignore_index=True)


def _filter_independent_cell_assignments_for_heatmap(
    *,
    adata: sc.AnnData,
    assignments: pd.DataFrame,
    heatmap_cell_ids: pd.Index,
    biocol: str,
    totals_by_type: pd.Series,
    threshold: float,
    allowed_cell_types: list[str] | None = None,
) -> pd.DataFrame:
    """
    Keep assignments only from trait × cell-type sections included in the heatmap.

    Inclusion is calculated from all marginal × conditional cells for the trait,
    using the same denominator and strict ``> threshold`` rule as the heatmap.
    Every assignment in a passing section is retained, including placeholder
    labels such as -1. ``allowed_cell_types`` can further restrict exports to
    the cell types that are actually present in a curated heatmap.
    """
    output_columns = ["marginal_x_conditional_cell_id", "independent_signal"]
    if assignments is None or len(assignments) == 0:
        return pd.DataFrame(columns=output_columns)

    missing_columns = [column for column in output_columns if column not in assignments.columns]
    if missing_columns:
        raise ValueError(f"Independent-cell assignments are missing columns: {missing_columns}")
    if biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs is missing '{biocol}'")

    celltype_order = totals_by_type.index.astype(str).tolist()
    section_fractions = discovery_fraction_by_celltype(
        adata=adata,
        discovered_cell_ids=heatmap_cell_ids,
        biocol=biocol,
        celltype_order=celltype_order,
        totals_by_type=totals_by_type,
    )
    passing_cell_types = set(
        section_fractions.index[
            section_fractions.map(lambda value: _passes_heatmap_inclusion(value, threshold))
        ].astype(str)
    )
    if allowed_cell_types is not None:
        passing_cell_types.intersection_update(str(cell_type) for cell_type in allowed_cell_types)
    if not passing_cell_types:
        return pd.DataFrame(columns=output_columns)

    work = assignments.loc[:, output_columns].copy()
    work["marginal_x_conditional_cell_id"] = work[
        "marginal_x_conditional_cell_id"
    ].astype(str)

    valid_cell_ids = adata.obs_names.intersection(
        work["marginal_x_conditional_cell_id"].astype(str)
    )
    work = work.loc[
        work["marginal_x_conditional_cell_id"].isin(valid_cell_ids)
    ].copy()
    if len(work) == 0:
        return pd.DataFrame(columns=output_columns)

    cell_type_by_id = adata.obs[biocol].astype(str)
    assignment_cell_types = work["marginal_x_conditional_cell_id"].map(cell_type_by_id)
    work = work.loc[assignment_cell_types.isin(passing_cell_types), output_columns]
    work = work.drop_duplicates(output_columns)
    work = work.sort_values(
        ["independent_signal", "marginal_x_conditional_cell_id"],
        kind="mergesort",
    ).reset_index(drop=True)
    return work


def build_scdrsfm_celltype_tables(
    *,
    adata: sc.AnnData,
    results_dir: Path,
    traits: Sequence[str],
    biocol: str,
    marginal_metacell_col: str = "metacell",
    fdr_alpha: float = 0.1,
    pval_col_candidates: Sequence[str] = ("pval", "mc_pval"),
    indep_sig_col: str = "independent_signal_multi",
    indep_cells_dir: Path | None = None,
    heatmap_threshold: float = 0.01,
    heatmap_traits: Sequence[str] | None = None,
    heatmap_cell_types: Sequence[str] | None = None,
    print_summaries: bool = False,
) -> dict[str, pd.DataFrame]:
    results_dir = Path(results_dir)
    if biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs missing {biocol!r}")
    if indep_cells_dir is not None:
        indep_cells_dir = Path(indep_cells_dir)
        indep_cells_dir.mkdir(parents=True, exist_ok=True)

    heatmap_threshold = float(heatmap_threshold)
    if not np.isfinite(heatmap_threshold) or not 0 <= heatmap_threshold <= 1:
        raise ValueError("heatmap_threshold must be a finite value between 0 and 1.")
    heatmap_trait_set = None if heatmap_traits is None else {str(trait) for trait in heatmap_traits}

    totals = celltype_counts(adata, biocol)
    celltype_order = totals.index.tolist()

    rows_marginal = {}
    rows_intersect = {}
    signal_details_rows = {}
    signal_count_rows = {}

    for trait in traits:
        prefix = Path(trait).name
        marginal_file = results_dir / f"{prefix}.marginal_score.gz"
        conditional_file = results_dir / f"{prefix}.conditional.tagging_score.gz"
        if not marginal_file.exists():
            raise FileNotFoundError(f"Missing marginal score file: {marginal_file}")
        if not conditional_file.exists():
            raise FileNotFoundError(f"Missing conditional score file: {conditional_file}")

        df_marg = pd.read_csv(marginal_file, sep="	", compression="infer", index_col=0)
        pcol_marg = pick_first_existing_col(df_marg, pval_col_candidates, what="marginal p-value")
        marg_sig = bh_fdr_mask(df_marg[pcol_marg].to_numpy(), alpha=fdr_alpha)
        marginal_sig_cells = adata.obs_names.intersection(df_marg.index[marg_sig].astype(str))

        df_cond = pd.read_csv(conditional_file, sep="	", compression="infer", index_col=0).copy()
        df_cond.index = pd.to_numeric(pd.Index(df_cond.index), errors="coerce")
        df_cond = df_cond.loc[df_cond.index.notna()]
        df_cond.index = df_cond.index.astype(int)

        if indep_sig_col not in df_cond.columns:
            raise ValueError(f"{conditional_file} missing {indep_sig_col!r}")
        if "cell_ids" not in df_cond.columns:
            raise ValueError(f"{conditional_file} missing 'cell_ids'")

        pcol_cond = pick_first_existing_col(df_cond, pval_col_candidates, what="conditional p-value")
        cond_sig = bh_fdr_mask(df_cond[pcol_cond].to_numpy(), alpha=fdr_alpha)
        df_cond_sig = df_cond.loc[cond_sig] if cond_sig.any() else df_cond.iloc[0:0]
        cond_sig_cells = adata.obs_names.intersection(cells_from_metacell_rows(df_cond_sig)) if len(df_cond_sig) else pd.Index([], dtype=str)
        marginal_x_cond_cells = marginal_sig_cells.intersection(cond_sig_cells)

        export_df_unfiltered = build_marginal_x_conditional_signal_assignments(
            adata=adata,
            marginal_sig_cells=marginal_sig_cells,
            df_cond_sig=df_cond_sig,
            indep_sig_col=indep_sig_col,
        )
        if heatmap_trait_set is not None and str(trait) not in heatmap_trait_set:
            export_df = pd.DataFrame(columns=["marginal_x_conditional_cell_id", "independent_signal"])
        else:
            export_df = _filter_independent_cell_assignments_for_heatmap(
                adata=adata,
                assignments=export_df_unfiltered,
                heatmap_cell_ids=marginal_x_cond_cells,
                biocol=biocol,
                totals_by_type=totals,
                threshold=heatmap_threshold,
                allowed_cell_types=None if heatmap_cell_types is None else list(heatmap_cell_types),
            )
        if indep_cells_dir is not None:
            export_df.to_csv(
                indep_cells_dir / f"{prefix}.gz",
                sep="	",
                index=False,
                compression="gzip",
            )

        rows_marginal[trait] = discovery_fraction_by_celltype(
            adata, marginal_sig_cells, biocol=biocol, celltype_order=celltype_order, totals_by_type=totals
        )
        rows_intersect[trait] = discovery_fraction_by_celltype(
            adata, marginal_x_cond_cells, biocol=biocol, celltype_order=celltype_order, totals_by_type=totals
        )

        all_signal_ids = valid_signal_values(df_cond[indep_sig_col])
        cond_signal_ids = valid_signal_values(df_cond_sig[indep_sig_col]) if len(df_cond_sig) else []
        n_signals_with_causal = 0

        for signal_id in all_signal_ids:
            signal_mask_all = pd.to_numeric(df_cond[indep_sig_col], errors="coerce").astype("Int64") == signal_id
            sub_all = df_cond.loc[signal_mask_all]
            all_signal_cells = adata.obs_names.intersection(cells_from_metacell_rows(sub_all)) if len(sub_all) else pd.Index([], dtype=str)
            marg_x_all_cells = marginal_sig_cells.intersection(all_signal_cells)

            if len(df_cond_sig):
                signal_mask_sig = pd.to_numeric(df_cond_sig[indep_sig_col], errors="coerce").astype("Int64") == signal_id
                sub_sig = df_cond_sig.loc[signal_mask_sig]
            else:
                sub_sig = df_cond_sig

            cond_signal_cells = adata.obs_names.intersection(cells_from_metacell_rows(sub_sig)) if len(sub_sig) else pd.Index([], dtype=str)
            marg_x_cond_signal_cells = marginal_sig_cells.intersection(cond_signal_cells)
            if len(marg_x_cond_signal_cells) > 0:
                n_signals_with_causal += 1

            signal_details_rows[(trait, int(signal_id))] = {
                "n_metacells_in_signal_all": int(len(sub_all)),
                "metacell_ids_in_signal_all": join_index(sub_all.index.astype(str)),
                "n_cells_in_signal_all": int(len(all_signal_cells)),
                "cell_ids_in_signal_all": join_index(all_signal_cells),
                "n_marg_x_signal_cells_all": int(len(marg_x_all_cells)),
                "marg_x_signal_cell_ids_all": join_index(marg_x_all_cells),
                "n_metacells_in_signal_cond_sig": int(len(sub_sig)),
                "metacell_ids_in_signal_cond_sig": join_index(sub_sig.index.astype(str)),
                "n_cells_in_signal_cond_sig": int(len(cond_signal_cells)),
                "cell_ids_in_signal_cond_sig": join_index(cond_signal_cells),
                "n_marg_x_signal_cells_cond_sig": int(len(marg_x_cond_signal_cells)),
                "marg_x_signal_cell_ids_cond_sig": join_index(marg_x_cond_signal_cells),
            }

        signal_count_rows[trait] = {
            "n_independent_signals_total": int(len(all_signal_ids)),
            "n_independent_signals_cond_sig": int(len(cond_signal_ids)),
            "n_independent_signals_with_any_causal_cells": int(n_signals_with_causal),
            "n_marginal_sig_cells": int(len(marginal_sig_cells)),
            "n_cond_sig_cells": int(len(cond_sig_cells)),
            "n_causal_cells_marg_x_cond": int(len(marginal_x_cond_cells)),
            "n_indep_cell_assignments_before_heatmap_filter": int(len(export_df_unfiltered)),
            "n_indep_cell_assignments_saved_after_heatmap_filter": int(len(export_df)),
        }
        if print_summaries:
            print(
                f"{trait}: {len(marginal_sig_cells):,} marginal cells; "
                f"{len(marginal_x_cond_cells):,} marginal∩conditional cells; "
                f"indep assignments raw={len(export_df_unfiltered):,}, "
                f"saved in sections >{heatmap_threshold:.1%}={len(export_df):,}"
            )

    df_marginal = pd.DataFrame.from_dict(rows_marginal, orient="index").astype(float)
    df_marginal.index.name = "trait"
    df_marginal.columns.name = biocol
    df_intersect = pd.DataFrame.from_dict(rows_intersect, orient="index").astype(float)
    df_intersect.index.name = "trait"
    df_intersect.columns.name = biocol
    df_counts = pd.DataFrame.from_dict(signal_count_rows, orient="index")
    df_counts.index.name = "trait"

    if signal_details_rows:
        df_details = pd.DataFrame.from_dict(signal_details_rows, orient="index")
        df_details.index = pd.MultiIndex.from_tuples(df_details.index, names=["trait", "independent_signal"])
    else:
        df_details = pd.DataFrame()
        df_details.index = pd.MultiIndex.from_tuples([], names=["trait", "independent_signal"])

    return {
        "df_marginal_props": df_marginal,
        "df_marginal_x_cond_props": df_intersect,
        "df_indep_signal_counts": df_counts,
        "df_signal_details": df_details,
    }

## Heatmap helpers

In [6]:
def display_group_for_celltype(cell_type: str, groups: Mapping[str, Sequence[str]]) -> str:
    for group, members in groups.items():
        if cell_type in members:
            return group
    return "Other"


def group_segments(cell_types: Sequence[str], groups: Mapping[str, Sequence[str]], colors: Mapping[str, str]):
    if not cell_types:
        return [], []
    group_names = [display_group_for_celltype(ct, groups) for ct in cell_types]
    seg_names = [group_names[0]]
    seg_counts = [0]
    for group in group_names:
        if group == seg_names[-1]:
            seg_counts[-1] += 1
        else:
            seg_names.append(group)
            seg_counts.append(1)
    color_counts = [(colors.get(group, "black"), count) for group, count in zip(seg_names, seg_counts)]
    return color_counts, seg_names


def aggregate_signal_cells(sub_trait: pd.DataFrame, raw_signal_id: object, *, signal_cell_ids_col: str) -> pd.Index:
    signal_ids = sub_trait.index.get_level_values(1).astype(str)
    rows = sub_trait.loc[signal_ids == str(raw_signal_id)]
    cells = pd.Index([], dtype=str)
    for value in rows[signal_cell_ids_col].astype(str).tolist():
        cells = cells.append(parse_csv_index(value))
    return pd.Index(cells.astype(str)).unique()


def build_independent_signal_annotations(
    *,
    adata: sc.AnnData,
    df_signal_details: pd.DataFrame | None,
    display_cell_types: Sequence[str],
    trait_index: Sequence[str],
    annotation_biocol: str,
    threshold: float,
    signal_cell_ids_col: str = "marg_x_signal_cell_ids_cond_sig",
    display_to_annotation_labels: Mapping[str, Sequence[str]] | None = None,
) -> tuple[dict[str, dict[str, str]], set[str]]:
    """Build white top-right independent-signal labels for plotted heatmap cells."""
    display_cell_types = [str(ct) for ct in display_cell_types]
    trait_index = [str(t) for t in trait_index]
    ann_map = {trait: {ct: "" for ct in display_cell_types} for trait in trait_index}
    no_discovery_traits: set[str] = set()

    if df_signal_details is None or len(df_signal_details) == 0:
        no_discovery_traits.update(trait_index)
        return ann_map, no_discovery_traits
    if not isinstance(df_signal_details.index, pd.MultiIndex):
        raise ValueError("df_signal_details must have a MultiIndex: trait × independent_signal")
    if signal_cell_ids_col not in df_signal_details.columns:
        raise ValueError(f"df_signal_details missing {signal_cell_ids_col!r}")
    if annotation_biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs missing {annotation_biocol!r}")

    totals = adata.obs[annotation_biocol].astype(str).value_counts()
    display_to_annotation_labels = display_to_annotation_labels or {ct: [ct] for ct in display_cell_types}
    available_traits = set(df_signal_details.index.get_level_values(0).astype(str))

    for trait in trait_index:
        if trait not in available_traits:
            no_discovery_traits.add(trait)
            continue
        sub_trait = df_signal_details.xs(trait, level=0, drop_level=False)
        if "n_marg_x_signal_cells_cond_sig" in sub_trait.columns:
            keep_any = pd.to_numeric(sub_trait["n_marg_x_signal_cells_cond_sig"], errors="coerce").fillna(0) > 0
        else:
            keep_any = sub_trait[signal_cell_ids_col].map(lambda x: len(parse_csv_index(x)) > 0)
        sub_any = sub_trait.loc[keep_any]
        if len(sub_any) == 0:
            no_discovery_traits.add(trait)
            continue

        raw_ids = sub_any.index.get_level_values(1).astype(str)
        numeric = pd.to_numeric(raw_ids, errors="coerce")
        if numeric.notna().all():
            sorted_raw_ids = [str(x) for x in sorted(numeric.astype(int).unique().tolist())]
        else:
            sorted_raw_ids = sorted(raw_ids.unique().tolist())

        raw_hits: dict[str, set[str]] = {}
        eligible: list[str] = []
        for raw_id in sorted_raw_ids:
            cells = aggregate_signal_cells(sub_any, raw_id, signal_cell_ids_col=signal_cell_ids_col)
            cells = adata.obs_names.intersection(cells.astype(str))
            if len(cells) == 0:
                continue
            counts = adata.obs.loc[cells, annotation_biocol].astype(str).value_counts()
            hits = set()
            for display_ct in display_cell_types:
                for label in display_to_annotation_labels.get(display_ct, [display_ct]):
                    denom = float(totals.get(label, 0))
                    if denom <= 0:
                        continue
                    if _passes_heatmap_inclusion(float(counts.get(label, 0)) / denom, threshold):
                        hits.add(display_ct)
                        break
            if hits:
                raw_hits[raw_id] = hits
                eligible.append(raw_id)

        if not eligible:
            no_discovery_traits.add(trait)
            continue
        remap = {raw_id: i + 1 for i, raw_id in enumerate(eligible)}
        for display_ct in display_cell_types:
            labels = [str(remap[raw_id]) for raw_id in eligible if display_ct in raw_hits.get(raw_id, set())]
            ann_map[trait][display_ct] = ",".join(labels)

    return ann_map, no_discovery_traits


def collapse_fine_props_to_parent_max(
    fine_props: pd.DataFrame,
    *,
    fine_to_parent: Mapping[str, str],
    parent_order: Sequence[str],
) -> pd.DataFrame:
    """For each trait × parent cell type, take the maximum value over its fine labels."""
    out = pd.DataFrame(0.0, index=fine_props.index, columns=list(parent_order))
    for fine_label, parent in fine_to_parent.items():
        if fine_label in fine_props.columns and parent in out.columns:
            out[parent] = np.maximum(out[parent].to_numpy(), fine_props[fine_label].to_numpy(dtype=float))
    return out.astype(float)


def fine_to_parent_map_from_adata(adata: sc.AnnData, *, fine_col: str, parent_col: str) -> dict[str, str]:
    tmp = pd.DataFrame({
        "fine": adata.obs[fine_col].astype(str).to_numpy(),
        "parent": adata.obs[parent_col].astype(str).to_numpy(),
    })
    counts = tmp.groupby(["fine", "parent"]).size().reset_index(name="n")
    counts = counts.sort_values(["fine", "n", "parent"], ascending=[True, False, True])
    return counts.drop_duplicates("fine").set_index("fine")["parent"].to_dict()




def _displayed_scdrsfm_annotation(
    *,
    trait: str,
    cell_type: str,
    conditional_star: bool,
    ann_map: Mapping[str, Mapping[str, str]],
    no_discovery_traits: set[str],
) -> str:
    if not conditional_star:
        return ""
    annotation = str(ann_map.get(str(trait), {}).get(str(cell_type), ""))
    if annotation:
        return annotation
    if str(trait) in no_discovery_traits:
        return "1"
    return ""


def _build_scdrsfm_heatmap_proportions_table(
    *,
    df_marginal: pd.DataFrame,
    df_conditional: pd.DataFrame,
    star_marginal: pd.DataFrame,
    star_conditional: pd.DataFrame,
    adata: sc.AnnData,
    biocol: str,
    trait_labels: Mapping[str, str],
    display_threshold: float,
    star_threshold: float,
    annotation_threshold: float,
    ann_map: Mapping[str, Mapping[str, str]],
    no_discovery_traits: set[str],
) -> pd.DataFrame:
    """Create a tidy table in the exact trait/cell-type order of a scDRS-FM heatmap."""
    frames = [df_marginal, df_conditional, star_marginal, star_conditional]
    if any(not frames[0].index.equals(frame.index) for frame in frames[1:]):
        raise ValueError("All heatmap matrices must have identical trait order.")
    if any(not frames[0].columns.equals(frame.columns) for frame in frames[1:]):
        raise ValueError("All heatmap matrices must have identical cell-type order.")

    traits = df_conditional.index.astype(str).tolist()
    cell_types = df_conditional.columns.astype(str).tolist()
    n_traits = len(traits)
    n_cell_types = len(cell_types)
    repeated_traits = np.repeat(np.asarray(traits, dtype=object), n_cell_types)
    tiled_cell_types = np.tile(np.asarray(cell_types, dtype=object), n_traits)

    marginal_values = df_marginal.astype(float).to_numpy().reshape(-1)
    conditional_values = df_conditional.astype(float).to_numpy().reshape(-1)
    star_marginal_values = star_marginal.astype(float).to_numpy().reshape(-1)
    star_conditional_values = star_conditional.astype(float).to_numpy().reshape(-1)

    display_pass = np.asarray([_passes_heatmap_inclusion(v, display_threshold) for v in conditional_values], dtype=bool)
    marginal_star = np.asarray([_passes_heatmap_inclusion(v, star_threshold) for v in star_marginal_values], dtype=bool)
    conditional_star = np.asarray([_passes_heatmap_inclusion(v, star_threshold) for v in star_conditional_values], dtype=bool)
    counts = celltype_counts(adata, biocol)

    annotations = [
        str(ann_map.get(str(trait), {}).get(str(cell_type), ""))
        for trait, cell_type in zip(repeated_traits, tiled_cell_types)
    ]
    displayed_annotations = [
        _displayed_scdrsfm_annotation(
            trait=str(trait),
            cell_type=str(cell_type),
            conditional_star=bool(star),
            ann_map=ann_map,
            no_discovery_traits=no_discovery_traits,
        )
        for trait, cell_type, star in zip(repeated_traits, tiled_cell_types, conditional_star)
    ]

    return pd.DataFrame({
        "trait_order": np.repeat(np.arange(1, n_traits + 1), n_cell_types),
        "cell_type_order": np.tile(np.arange(1, n_cell_types + 1), n_traits),
        "trait": repeated_traits,
        "trait_label": [trait_labels.get(str(trait), str(trait)) for trait in repeated_traits],
        "cell_type": tiled_cell_types,
        "cell_type_label": tiled_cell_types,
        "cell_type_total_cells": [int(counts.get(str(cell_type), 0)) for cell_type in tiled_cell_types],
        "marginal_cell_type_proportion": marginal_values,
        "conditional_cell_type_proportion": conditional_values,
        "conditional_proportion_displayed": np.where(display_pass, conditional_values, 0.0),
        "conditional_passes_display_threshold": display_pass,
        "star_marginal_cell_type_proportion": star_marginal_values,
        "marginal_star_displayed": marginal_star,
        "star_conditional_cell_type_proportion": star_conditional_values,
        "conditional_star_displayed": conditional_star,
        "independent_signals_passing_annotation_threshold": annotations,
        "independent_signal_annotation_displayed": displayed_annotations,
        "display_threshold": float(display_threshold),
        "star_threshold": float(star_threshold),
        "annotation_threshold": float(annotation_threshold),
        "display_inclusion_rule": f"> {float(display_threshold):g}",
        "star_inclusion_rule": f"> {float(star_threshold):g}",
        "annotation_inclusion_rule": f"> {float(annotation_threshold):g}",
    })


def _write_scdrsfm_heatmap_proportions_csv(*, out_csv: Path | str, **kwargs) -> pd.DataFrame:
    table = _build_scdrsfm_heatmap_proportions_table(**kwargs)
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(out_csv, index=False)
    print(f"Saved heatmap cell-type proportions: {out_csv} ({len(table):,} rows)")
    return table


def plot_scdrsfm_heatmap(
    *,
    df_marginal_props: pd.DataFrame,
    df_intersect_props: pd.DataFrame,
    adata: sc.AnnData,
    biocol: str,
    trait_order: Sequence[str],
    trait_labels: Mapping[str, str],
    threshold: float = 0.01,
    out_png: Path | str,
    out_csv: Path | str | None = None,
    title: str = "",
    df_signal_details: pd.DataFrame | None = None,
    signal_cell_ids_col: str = "marg_x_signal_cell_ids_cond_sig",
    celltype_order: Sequence[str] | None = None,
    celltype_groups: Mapping[str, Sequence[str]] | None = None,
    celltype_group_colors: Mapping[str, str] | None = None,
    star_marginal_props: pd.DataFrame | None = None,
    star_conditional_props: pd.DataFrame | None = None,
    star_threshold: float | None = None,
    annotation_biocol: str | None = None,
    annotation_threshold: float | None = None,
    display_to_annotation_labels: Mapping[str, Sequence[str]] | None = None,
    fontsize_mult: float = 1.0,
    cbar_label: str = "Prop. sig. conditional cells",
) -> list[str]:
    """Plot a square-cell scDRS-FM heatmap."""
    trait_order = [t for t in trait_order if t in df_marginal_props.index and t in df_intersect_props.index]
    if not trait_order:
        raise ValueError("No requested traits are present in the heatmap matrices.")

    df_marg = df_marginal_props.loc[trait_order].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    df_cond = df_intersect_props.loc[trait_order].apply(pd.to_numeric, errors="coerce").fillna(0.0)

    star_threshold = threshold if star_threshold is None else float(star_threshold)
    star_marg = (star_marginal_props.loc[trait_order] if star_marginal_props is not None else df_marg).apply(pd.to_numeric, errors="coerce").fillna(0.0)
    star_cond = (star_conditional_props.loc[trait_order] if star_conditional_props is not None else df_cond).apply(pd.to_numeric, errors="coerce").fillna(0.0)

    common_cols = df_marg.columns.intersection(df_cond.columns).intersection(star_marg.columns).intersection(star_cond.columns)
    df_marg = df_marg[common_cols]
    df_cond = df_cond[common_cols]
    star_marg = star_marg[common_cols]
    star_cond = star_cond[common_cols]

    keep_cols = star_marg.columns[(star_marg > star_threshold).any(axis=0)].tolist()
    if celltype_order is not None:
        ordered_cols = [ct for ct in celltype_order if ct in keep_cols]
        ordered_cols.extend(sorted(ct for ct in keep_cols if ct not in set(ordered_cols)))
    else:
        ordered_cols = keep_cols
    if not ordered_cols:
        raise ValueError("No cell types passed the marginal association filter.")

    df_marg = df_marg[ordered_cols]
    df_cond = df_cond[ordered_cols]
    star_marg = star_marg[ordered_cols]
    star_cond = star_cond[ordered_cols]

    annotation_biocol = annotation_biocol or biocol
    annotation_threshold = threshold if annotation_threshold is None else float(annotation_threshold)
    ann_map, no_discovery_traits = build_independent_signal_annotations(
        adata=adata,
        df_signal_details=df_signal_details,
        display_cell_types=ordered_cols,
        trait_index=trait_order,
        annotation_biocol=annotation_biocol,
        threshold=annotation_threshold,
        signal_cell_ids_col=signal_cell_ids_col,
        display_to_annotation_labels=display_to_annotation_labels,
    )

    n_rows, n_cols = df_cond.shape
    fig, ax, bbox = create_square_heatmap_figure(
        n_rows, n_cols,
        cell_size=0.72,
        left_margin=7.0,
        right_margin=3.0,
        bottom_margin=7.0,
        top_margin=4.5,
    )
    ax_left, ax_bottom, ax_width, ax_height = bbox

    boundaries = np.linspace(0, 1, 12)
    cmap = ListedColormap(sns.color_palette("Blues", n_colors=11))
    norm = BoundaryNorm(boundaries, ncolors=cmap.N, clip=True)

    for i, trait in enumerate(trait_order):
        y = n_rows - i - 1
        for j, cell_type in enumerate(ordered_cols):
            raw_cond = float(df_cond.loc[trait, cell_type])
            raw_marg_star = float(star_marg.loc[trait, cell_type])
            raw_cond_star = float(star_cond.loc[trait, cell_type])
            display_value = raw_cond if _passes_heatmap_inclusion(raw_cond, threshold) else 0.0
            facecolor = "white" if display_value == 0.0 else cmap(norm(display_value))
            ax.add_patch(Rectangle((j, y), 1, 1, facecolor=facecolor, edgecolor="none"))

            if _passes_heatmap_inclusion(raw_marg_star, star_threshold):
                ax.text(j + 0.5, y + 0.5, "☆", ha="center", va="center",
                        fontsize=42 * fontsize_mult, color="black", fontweight="bold", zorder=20)
            if _passes_heatmap_inclusion(raw_cond_star, star_threshold):
                ax.text(j + 0.5, y + 0.5, "★", ha="center", va="center",
                        fontsize=34 * fontsize_mult, color="red", fontweight="bold",
                        path_effects=[pe.withStroke(linewidth=2.0, foreground="black")], zorder=21)
                ann = ann_map.get(trait, {}).get(cell_type, "")
                if ann:
                    ax.text(j + 0.95, y + 0.95, ann, ha="right", va="top",
                            fontsize=15 * fontsize_mult, color="white", fontweight="bold",
                            path_effects=[pe.withStroke(linewidth=2.8, foreground="black")], zorder=30)
                elif trait in no_discovery_traits:
                    ax.text(j + 0.95, y + 0.95, "1", ha="right", va="top",
                            fontsize=15 * fontsize_mult, color="white", fontweight="bold",
                            path_effects=[pe.withStroke(linewidth=2.8, foreground="black")], zorder=30)

    ax.set_xlim(0, n_cols)
    ax.set_ylim(0, n_rows)
    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_xticks(np.arange(n_cols + 1), minor=True)
    ax.set_yticks(np.arange(n_rows + 1), minor=True)
    ax.grid(which="minor", color="lightgray", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    counts = celltype_counts(adata, biocol)
    ax.set_xticklabels(labels_with_counts(ordered_cols, counts), rotation=45, ha="right", fontsize=18 * fontsize_mult)
    y_labels = [trait_labels.get(t, t) for t in reversed(trait_order)]
    ax.set_yticklabels(y_labels, fontsize=20 * fontsize_mult)

    if celltype_groups is not None:
        colors = celltype_group_colors or {}
        tick_colors = [colors.get(display_group_for_celltype(ct, celltype_groups), "black") for ct in ordered_cols]
        color_ticklabels(ax.get_xticklabels(), tick_colors, fontsize=18 * fontsize_mult, rotation=45, ha="right")
        color_counts, names = group_segments(ordered_cols, celltype_groups, colors)
        add_top_group_lines(ax, color_counts, names, fontsize=24 * fontsize_mult)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar_ax = fig.add_axes([ax_left, min(0.96, ax_bottom + ax_height + 0.22), min(0.35, ax_width * 0.7), 0.018])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal", boundaries=boundaries,
                        ticks=[0, 0.5, 1.0], spacing="proportional", drawedges=True)
    cbar.ax.set_xticklabels(["0%", "50%", "100%"], fontsize=15 * fontsize_mult)
    cbar.set_label(cbar_label, fontsize=18 * fontsize_mult, labelpad=12 * fontsize_mult)

    inferred_handle = Line2D([], [], linestyle="None", marker="$1$", color="white", markersize=15 * fontsize_mult)
    inferred_handle.set_path_effects([pe.withStroke(linewidth=2.8, foreground="black")])
    legend_elements = [
        Line2D([], [], marker="*", linestyle="None", markerfacecolor="none", markeredgecolor="black",
               markeredgewidth=2, markersize=22 * fontsize_mult),
        Line2D([], [], marker="*", linestyle="None", markerfacecolor="red", markeredgecolor="black",
               markersize=22 * fontsize_mult),
        inferred_handle,
    ]
    fig.legend(
        legend_elements,
        ["Marginal association", "Conditional association", "Inferred cell population"],
        loc="upper right",
        bbox_to_anchor=(0.985, 0.985),
        prop={"size": 16 * fontsize_mult},
        frameon=True,
    )
    if title:
        fig.suptitle(title, fontsize=24 * fontsize_mult, y=0.995)

    plt.savefig(out_png, bbox_inches="tight", dpi=300)
    if out_csv is not None:
        _write_scdrsfm_heatmap_proportions_csv(
            out_csv=out_csv,
            df_marginal=df_marg,
            df_conditional=df_cond,
            star_marginal=star_marg,
            star_conditional=star_cond,
            adata=adata,
            biocol=biocol,
            trait_labels=trait_labels,
            display_threshold=threshold,
            star_threshold=star_threshold,
            annotation_threshold=annotation_threshold,
            ann_map=ann_map,
            no_discovery_traits=no_discovery_traits,
        )
    plt.show()
    print(f"Saved: {out_png}")
    return ordered_cols

## scDRS and scPagwas parsing / heatmaps

In [7]:

def choose_scdrs_score_file(
    results_dir: Path,
    trait: str,
    *,
    suffix_candidates: Sequence[str],
) -> Path:
    """Return the first existing scDRS/scDRS-FM score file for a trait."""
    prefix = Path(trait).name
    candidates = [Path(results_dir) / f"{prefix}{suffix}" for suffix in suffix_candidates]
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


def build_marginal_props_from_score_files(
    *,
    adata: sc.AnnData,
    results_dir: Path,
    traits: Sequence[str],
    biocol: str,
    fdr_alpha: float = 0.1,
    pval_col_candidates: Sequence[str] = ("pval", "mc_pval"),
    score_suffix: str | None = ".score.gz",
    score_suffix_candidates: Sequence[str] | None = None,
) -> pd.DataFrame:
    """
    Build trait × cell-type fractions from significant single-cell score files.

    This is used for plain scDRS and other marginal-only methods. A cell is
    significant if its score-file p-value passes BH-FDR at `fdr_alpha`.
    """
    totals = celltype_counts(adata, biocol)
    celltype_order = totals.index.tolist()
    rows: dict[str, pd.Series] = {}

    if score_suffix_candidates is None:
        score_suffix_candidates = (score_suffix or ".score.gz",)

    for trait in traits:
        score_file = choose_scdrs_score_file(
            Path(results_dir),
            trait,
            suffix_candidates=score_suffix_candidates,
        )
        if not score_file.exists():
            raise FileNotFoundError(f"Missing score file for {trait}: {score_file}")

        df = pd.read_csv(score_file, sep="\t", compression="infer", index_col=0)
        pcol = pick_first_existing_col(df, pval_col_candidates, what=f"{score_file} p-value")

        sig_mask = bh_fdr_mask(df[pcol].to_numpy(), alpha=fdr_alpha)
        sig_cells = adata.obs_names.intersection(df.index[sig_mask].astype(str))

        rows[trait] = discovery_fraction_by_celltype(
            adata,
            sig_cells,
            biocol=biocol,
            celltype_order=celltype_order,
            totals_by_type=totals,
        )

    props = pd.DataFrame.from_dict(rows, orient="index").astype(float)
    props.index.name = "trait"
    props.columns.name = biocol
    return props


def build_threshold_assoc_from_props(
    props: pd.DataFrame,
    *,
    threshold: float = 0.05,
) -> dict[str, list[str]]:
    """
    Convert a trait × cell-type proportion table to association calls.

    Used for SEA-AD scDRS validation where a Region_Supertype is called
    associated when >5% of that Region_Supertype's cells are significant.
    """
    assoc: dict[str, list[str]] = {}
    props_num = props.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    for trait in props_num.index.astype(str):
        assoc[trait] = props_num.columns[(props_num.loc[trait] > threshold)].astype(str).tolist()

    return assoc


def build_celltype_assoc_from_scdrs_ct_files(
    *,
    results_dir: Path,
    traits: Sequence[str],
    biocol: str,
    alpha: float = 0.05,
    pval_col: str = "assoc_mcp",
    file_suffix_template: str = ".scdrs_ct.{biocol}",
) -> dict[str, list[str]]:
    """
    Cell-type associations from scDRS cell-type association files.

    Kept for compatibility, but the SEA-AD scDRS heatmap below uses the
    requested >5% Region_Supertype-cell threshold instead.
    """
    assoc: dict[str, list[str]] = {}

    for trait in traits:
        prefix = Path(trait).name
        suffix = file_suffix_template.format(biocol=biocol)
        ct_file = Path(results_dir) / f"{prefix}{suffix}"

        if not ct_file.exists():
            assoc[trait] = []
            print(f"Warning: missing scDRS cell-type file: {ct_file}")
            continue

        df = pd.read_csv(ct_file, sep="\t", index_col=0)
        if pval_col not in df.columns:
            raise ValueError(f"{ct_file} missing {pval_col!r}; columns={df.columns.tolist()}")

        sig = bh_fdr_mask(df[pval_col].to_numpy(), alpha=alpha)
        assoc[trait] = df.index[sig].astype(str).tolist()

    return assoc


def scpagwas_trait_dir(base_dir: Path, trait: str) -> Path:
    """
    Return the per-trait scPagwas directory.

    Supports either the parent directory:
        scpagwas_traits/output_newdata/sea_ad
    or an already trait-specific directory:
        scpagwas_traits/output_newdata/sea_ad/PASS_Alzheimers_Jansen2019
    """
    base_dir = Path(base_dir)
    prefix = Path(trait).name
    return base_dir if base_dir.name == prefix else base_dir / prefix


def scpagwas_singlecell_file(base_dir: Path, trait: str) -> Path:
    prefix = Path(trait).name
    trait_dir = scpagwas_trait_dir(base_dir, trait)

    candidates = [
        trait_dir / f"{prefix}_snglecell_scPagwas_score_pvalue.Result.csv",
        trait_dir / f"{prefix}_singlecell_scPagwas_score_pvalue.Result.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


def scpagwas_celltype_file(base_dir: Path, trait: str) -> Path:
    prefix = Path(trait).name
    return scpagwas_trait_dir(base_dir, trait) / f"{prefix}_Merged_celltype_pvalue.csv"


def build_scpagwas_tables(
    *,
    adata: sc.AnnData,
    base_dir: Path,
    traits: Sequence[str],
    biocol: str,
    cell_alpha: float = 0.1,
    ct_alpha: float = 0.05,
    score_col: str = "scPagwas.TRS.Score",
    score_sig_col: str = "Random_Correct_BG_adjp",
    ct_pval_col: str = "pvalue",
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """
    Build scPagwas heatmap inputs from the single-cell and merged cell-type files.

    Cells:
        significant if Random_Correct_BG_adjp < `cell_alpha`

    Cell types:
        significant if BH-FDR on `_Merged_celltype_pvalue.csv` p-values
        passes `ct_alpha`
    """
    totals = celltype_counts(adata, biocol)
    celltype_order = totals.index.tolist()
    prop_rows: dict[str, pd.Series] = {}
    assoc: dict[str, list[str]] = {}

    for trait in traits:
        score_file = scpagwas_singlecell_file(base_dir, trait)
        ct_file = scpagwas_celltype_file(base_dir, trait)

        if not score_file.exists():
            raise FileNotFoundError(f"Missing scPagwas single-cell file for {trait}: {score_file}")
        if not ct_file.exists():
            raise FileNotFoundError(f"Missing scPagwas cell-type file for {trait}: {ct_file}")

        df_score = pd.read_csv(score_file, index_col=0)
        if score_sig_col not in df_score.columns:
            raise ValueError(f"{score_file} missing {score_sig_col!r}; columns={df_score.columns.tolist()}")
        if score_col not in df_score.columns:
            print(f"Warning: {score_file} missing {score_col!r}; using {score_sig_col!r} only for heatmap fractions.")

        sig_cells = adata.obs_names.intersection(
            df_score.index[
                pd.to_numeric(df_score[score_sig_col], errors="coerce") < cell_alpha
            ].astype(str)
        )
        prop_rows[trait] = discovery_fraction_by_celltype(
            adata,
            sig_cells,
            biocol=biocol,
            celltype_order=celltype_order,
            totals_by_type=totals,
        )

        df_ct = pd.read_csv(ct_file, index_col=0)
        if "celltype" not in df_ct.columns or ct_pval_col not in df_ct.columns:
            raise ValueError(
                f"{ct_file} must contain 'celltype' and {ct_pval_col!r}; "
                f"columns={df_ct.columns.tolist()}"
            )

        sig_ct = bh_fdr_mask(pd.to_numeric(df_ct[ct_pval_col], errors="coerce").to_numpy(), alpha=ct_alpha)
        assoc[trait] = df_ct.loc[sig_ct, "celltype"].astype(str).tolist()

    props = pd.DataFrame.from_dict(prop_rows, orient="index").astype(float)
    props.index.name = "trait"
    props.columns.name = biocol
    return props, assoc


def _associated_lookup(
    associated_celltypes: Mapping[str, Sequence[str]] | pd.DataFrame,
    trait: str,
) -> set[str]:
    """Return associated cell types for a trait from either dict or bool matrix."""
    if isinstance(associated_celltypes, pd.DataFrame):
        if trait not in associated_celltypes.index:
            return set()
        row = associated_celltypes.loc[trait].astype(bool)
        return set(row.index[row].astype(str))
    return set(str(ct) for ct in associated_celltypes.get(trait, []))




def _build_marginal_method_heatmap_proportions_table(
    *,
    df_props: pd.DataFrame,
    associated_celltypes: Mapping[str, Sequence[str]] | pd.DataFrame,
    adata: sc.AnnData,
    biocol: str,
    trait_labels: Mapping[str, str],
    threshold: float,
) -> pd.DataFrame:
    traits = df_props.index.astype(str).tolist()
    cell_types = df_props.columns.astype(str).tolist()
    n_traits = len(traits)
    n_cell_types = len(cell_types)
    repeated_traits = np.repeat(np.asarray(traits, dtype=object), n_cell_types)
    tiled_cell_types = np.tile(np.asarray(cell_types, dtype=object), n_traits)
    values = df_props.astype(float).to_numpy().reshape(-1)
    displayed = np.asarray([_passes_heatmap_inclusion(value, threshold) for value in values], dtype=bool)
    associated = np.asarray([
        str(cell_type) in _associated_lookup(associated_celltypes, str(trait))
        for trait, cell_type in zip(repeated_traits, tiled_cell_types)
    ], dtype=bool)
    counts = celltype_counts(adata, biocol)
    return pd.DataFrame({
        "trait_order": np.repeat(np.arange(1, n_traits + 1), n_cell_types),
        "cell_type_order": np.tile(np.arange(1, n_cell_types + 1), n_traits),
        "trait": repeated_traits,
        "trait_label": [trait_labels.get(str(trait), str(trait)) for trait in repeated_traits],
        "cell_type": tiled_cell_types,
        "cell_type_label": tiled_cell_types,
        "cell_type_total_cells": [int(counts.get(str(cell_type), 0)) for cell_type in tiled_cell_types],
        "cell_type_proportion": values,
        "cell_type_proportion_displayed": np.where(displayed, values, 0.0),
        "passes_display_threshold": displayed,
        "association_star_displayed": associated,
        "display_threshold": float(threshold),
        "display_inclusion_rule": f"> {float(threshold):g}",
    })


def _write_marginal_method_heatmap_proportions_csv(*, out_csv: Path | str, **kwargs) -> pd.DataFrame:
    table = _build_marginal_method_heatmap_proportions_table(**kwargs)
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(out_csv, index=False)
    print(f"Saved heatmap cell-type proportions: {out_csv} ({len(table):,} rows)")
    return table


def plot_marginal_method_heatmap(
    *,
    df_props: pd.DataFrame,
    associated_celltypes: Mapping[str, Sequence[str]] | pd.DataFrame,
    adata: sc.AnnData,
    biocol: str,
    trait_order: Sequence[str],
    trait_labels: Mapping[str, str],
    out_png: Path | str,
    out_csv: Path | str | None = None,
    title: str = "",
    threshold: float = 0.0,
    celltype_order: Sequence[str] | None = None,
    celltype_groups: Mapping[str, Sequence[str]] | None = None,
    celltype_group_colors: Mapping[str, str] | None = None,
    cbar_label: str = "Prop. sig. marginal cells",
    star_label: str = "Cell-type association",
    fontsize_mult: float = 1.0,
) -> list[str]:
    """Plot a marginal-only heatmap with square cells and red association stars."""
    trait_order = [t for t in trait_order if t in df_props.index]
    if not trait_order:
        raise ValueError("No traits available for plotting.")

    df = df_props.loc[trait_order].apply(pd.to_numeric, errors="coerce").fillna(0.0)

    assoc_cols: set[str] = set()
    for trait in trait_order:
        assoc_cols.update(_associated_lookup(associated_celltypes, trait))

    assoc_cols = [ct for ct in df.columns if ct in assoc_cols]

    # If the external association labels do not match the adata cell-type labels,
    # still show any columns with nonzero significant-cell fractions.
    if not assoc_cols:
        assoc_cols = df.columns[(df > threshold).any(axis=0)].astype(str).tolist()

    if celltype_order is not None:
        ordered_cols = [ct for ct in celltype_order if ct in assoc_cols]
        ordered_cols.extend(sorted(ct for ct in assoc_cols if ct not in set(ordered_cols)))
    else:
        ordered_cols = sorted(assoc_cols)

    if not ordered_cols:
        raise ValueError("No cell types available for marginal-method heatmap.")

    df = df.loc[:, ordered_cols]
    n_rows, n_cols = df.shape

    fig, ax, bbox = create_square_heatmap_figure(
        n_rows,
        n_cols,
        cell_size=0.72,
        left_margin=7.0,
        right_margin=3.0,
        bottom_margin=7.0,
        top_margin=4.5,
    )
    ax_left, ax_bottom, ax_width, ax_height = bbox

    boundaries = np.linspace(0, 1, 12)
    cmap = ListedColormap(sns.color_palette("Blues", n_colors=11))
    norm = BoundaryNorm(boundaries, ncolors=cmap.N, clip=True)

    for i, trait in enumerate(trait_order):
        y = n_rows - i - 1
        assoc_set = _associated_lookup(associated_celltypes, trait)

        for j, cell_type in enumerate(ordered_cols):
            value = float(df.loc[trait, cell_type])
            display_value = 0.0 if value <= threshold else value
            facecolor = "white" if display_value == 0.0 else cmap(norm(display_value))
            ax.add_patch(Rectangle((j, y), 1, 1, facecolor=facecolor, edgecolor="none"))

            if cell_type in assoc_set:
                ax.text(
                    j + 0.5,
                    y + 0.5,
                    "★",
                    ha="center",
                    va="center",
                    fontsize=34 * fontsize_mult,
                    color="red",
                    fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2.0, foreground="black")],
                    zorder=10,
                )

    ax.set_xlim(0, n_cols)
    ax.set_ylim(0, n_rows)
    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_xticks(np.arange(n_cols + 1), minor=True)
    ax.set_yticks(np.arange(n_rows + 1), minor=True)
    ax.grid(which="minor", color="lightgray", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    ax.set_xticklabels(
        labels_with_counts(ordered_cols, celltype_counts(adata, biocol)),
        rotation=45,
        ha="right",
        fontsize=18 * fontsize_mult,
    )
    traits_bottom_to_top = list(reversed(trait_order))
    ax.set_yticklabels(
        [trait_labels.get(t, t) for t in traits_bottom_to_top],
        fontsize=20 * fontsize_mult,
    )

    if celltype_groups is not None:
        colors = celltype_group_colors or {}
        tick_colors = [
            colors.get(display_group_for_celltype(ct, celltype_groups), "black")
            for ct in ordered_cols
        ]
        color_ticklabels(
            ax.get_xticklabels(),
            tick_colors,
            fontsize=18 * fontsize_mult,
            rotation=45,
            ha="right",
        )
        color_counts, names = group_segments(ordered_cols, celltype_groups, colors)
        add_top_group_lines(ax, color_counts, names, fontsize=24 * fontsize_mult)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar_ax = fig.add_axes([
        ax_left,
        min(0.96, ax_bottom + ax_height + 0.22),
        min(0.35, ax_width * 0.7),
        0.018,
    ])
    cbar = fig.colorbar(
        sm,
        cax=cbar_ax,
        orientation="horizontal",
        boundaries=boundaries,
        ticks=[0, 0.5, 1.0],
        spacing="proportional",
        drawedges=True,
    )
    cbar.ax.set_xticklabels(["0%", "50%", "100%"], fontsize=15 * fontsize_mult)
    cbar.set_label(cbar_label, fontsize=18 * fontsize_mult, labelpad=12 * fontsize_mult)

    legend_elements = [
        Line2D(
            [],
            [],
            marker="*",
            linestyle="None",
            markerfacecolor="red",
            markeredgecolor="black",
            markersize=20 * fontsize_mult,
            label=star_label,
        )
    ]
    fig.legend(
        legend_elements,
        [star_label],
        loc="upper right",
        bbox_to_anchor=(0.985, 0.985),
        prop={"size": 16 * fontsize_mult},
        frameon=True,
    )

    if title:
        fig.suptitle(title, fontsize=24 * fontsize_mult, y=0.995)

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, bbox_inches="tight", dpi=300)
    if out_csv is not None:
        _write_marginal_method_heatmap_proportions_csv(
            out_csv=out_csv,
            df_props=df,
            associated_celltypes=associated_celltypes,
            adata=adata,
            biocol=biocol,
            trait_labels=trait_labels,
            threshold=threshold,
        )
    plt.show()
    print(f"Saved: {out_png}")
    return ordered_cols


In [8]:
# ============================================================
# Corrected scPagwas / scDRS helper overrides
# ============================================================
# Fixes:
#   1. scPagwas cell IDs are aligned to SEA-AD adata IDs by removing the
#      final adata-only suffix, e.g. "...-1003930257-A9" -> "...-1003930257".
#   2. scPagwas merged cell-type labels are mapped to SEA-AD Subclass labels.
#   3. scDRS heatmap stars are based on >5% significant cells within any
#      Region_Supertype, then projected back to the parent Subclass.
#   4. scDRS-FM trait score UMAPs highlight marginal × conditional significant cells.
#   5. Independent-population UMAP legend matches the T-cell notebook style.

import re
from collections import defaultdict


# Root containing per-trait scPagwas folders.
# Existing notebook defines SEA_AD_SCPAGWAS_AD_DIR as:
#   scpagwas_traits/output_newdata/sea_reference/PASS_Alzheimers_Jansen2019
SEA_AD_SCPAGWAS_ROOT = Path(SEA_AD_SCPAGWAS_AD_DIR).parent


# Expanded SEA-AD Subclass group/order definitions.
# These include the scPagwas merged-celltype labels after mapping.
SEA_SUBCLASS_GROUPS = {
    "Glial": [
        "Astrocyte",
        "Endothelial",
        "Microglia-PVM",
        "Oligodendrocyte",
        "OPC",
        "VLMC",
    ],
    "Excitatory": [
        "L2/3 IT",
        "L4 IT",
        "L5 ET",
        "L5 IT",
        "L5/6 NP",
        "L6 CT",
        "L6 IT",
        "L6 IT Car3",
        "L6b",
    ],
    "Inhibitory": [
        "Chandelier",
        "Lamp5",
        "Lamp5 Lhx6",
        "Pax6",
        "Pvalb",
        "Sncg",
        "Sst",
        "Sst Chodl",
        "Vip",
    ],
}
SEA_GROUP_COLORS = {
    "Glial": "green",
    "Excitatory": "red",
    "Inhibitory": "blue",
    "Other": "black",
}
SEA_SUBCLASS_ORDER = [ct for group in SEA_SUBCLASS_GROUPS.values() for ct in group]


SCPAGWAS_CELLTYPE_TO_SEAAD_SUBCLASS = {
    "Astro": "Astrocyte",
    "Chandelier": "Chandelier",
    "Endo": "Endothelial",
    "L2/3 IT": "L2/3 IT",
    "L4 IT": "L4 IT",
    "L5 ET": "L5 ET",
    "L5 IT": "L5 IT",
    "L5/6 NP": "L5/6 NP",
    "L6 CT": "L6 CT",
    "L6 IT": "L6 IT",
    "L6 IT Car3": "L6 IT Car3",
    "L6b": "L6b",
    "Lamp5": "Lamp5",
    "Lamp5_Lhx6": "Lamp5 Lhx6",
    "Lamp5 Lhx6": "Lamp5 Lhx6",
    "Micro-PVM": "Microglia-PVM",
    "Oligo": "Oligodendrocyte",
    "OPC": "OPC",
    "Pax6": "Pax6",
    "Pvalb": "Pvalb",
    "Sncg": "Sncg",
    "Sst": "Sst",
    "Sst Chodl": "Sst Chodl",
    "Vip": "Vip",
    "VLMC": "VLMC",
}


def canonical_seaad_cell_id(cell_id: str) -> str:
    """
    Canonical SEA-AD cell ID used to align external scPagwas files.

    SEA-AD adata IDs often have one extra final suffix:
        TCGA...-1003930257-A9

    scPagwas IDs usually omit that suffix:
        TCGA...-1003930257

    This removes only simple final suffixes like -A9, -B12, etc.
    """
    s = str(cell_id).strip()
    if not s or s == "nan":
        return s

    left, sep, last = s.rpartition("-")
    if sep and re.fullmatch(r"[A-Za-z]+[0-9]+", last):
        return left

    return s


def build_external_to_adata_cell_map(
    external_ids,
    adata_obs_names,
    *,
    verbose: bool = True,
) -> dict[str, str]:
    """
    Map external cell IDs to adata.obs_names.

    Matching priority:
      1. exact external ID == adata.obs_name,
      2. canonicalized ID match, where adata's final suffix is stripped.
    """
    external_ids = pd.Index(external_ids.astype(str) if hasattr(external_ids, "astype") else [str(x) for x in external_ids])
    adata_obs_names = pd.Index(adata_obs_names.astype(str) if hasattr(adata_obs_names, "astype") else [str(x) for x in adata_obs_names])

    adata_exact = set(adata_obs_names)

    canonical_to_adata = defaultdict(list)
    for obs_name in adata_obs_names:
        canonical_to_adata[canonical_seaad_cell_id(obs_name)].append(obs_name)

    out = {}
    duplicate_canonical_matches = 0

    for ext_id in external_ids:
        if ext_id in adata_exact:
            out[ext_id] = ext_id
            continue

        canonical = canonical_seaad_cell_id(ext_id)
        hits = canonical_to_adata.get(canonical, [])

        if len(hits) == 1:
            out[ext_id] = hits[0]
        elif len(hits) > 1:
            # Should be rare. Keep the first deterministic match and report it.
            out[ext_id] = sorted(hits)[0]
            duplicate_canonical_matches += 1

    if verbose:
        print(
            f"External cell ID alignment: matched {len(out):,}/{len(external_ids):,} "
            f"external IDs to {len(set(out.values())):,} adata cells."
        )
        if duplicate_canonical_matches:
            print(f"  Warning: {duplicate_canonical_matches:,} external IDs had duplicate canonical adata matches.")

    return out


def align_external_cell_table_to_adata(
    df: pd.DataFrame,
    adata: sc.AnnData,
    *,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Align an external single-cell table to adata.obs_names using robust SEA-AD IDs.

    Returns a dataframe indexed by adata.obs_names. Unmatched adata cells are NaN.
    """
    df = df.copy()
    df.index = df.index.astype(str)

    external_to_adata = build_external_to_adata_cell_map(
        df.index,
        adata.obs_names.astype(str),
        verbose=verbose,
    )

    if len(external_to_adata) == 0:
        raise ValueError(
            "No external cell IDs matched adata.obs_names. "
            "Check whether the scPagwas file belongs to this SEA-AD AnnData."
        )

    matched = df.loc[list(external_to_adata.keys())].copy()
    matched.index = [external_to_adata[x] for x in matched.index]

    # If there are accidental duplicates after canonical matching, keep the first.
    matched = matched.loc[~matched.index.duplicated(keep="first")]

    return matched.reindex(adata.obs_names.astype(str))


def normalize_celltype_label(label: str) -> str:
    """Normalize labels for case/punctuation-insensitive matching."""
    return re.sub(r"[^a-z0-9]+", "", str(label).lower())


def map_scpagwas_celltype_to_adata_label(
    celltype: str,
    available_labels,
    *,
    explicit_map: Mapping[str, str] = SCPAGWAS_CELLTYPE_TO_SEAAD_SUBCLASS,
) -> str | None:
    """
    Map scPagwas merged-celltype labels to SEA-AD adata Subclass labels.
    """
    available_labels = [str(x) for x in available_labels]
    available_set = set(available_labels)

    raw = str(celltype).strip()
    candidate = explicit_map.get(raw, raw.replace("_", " "))

    if candidate in available_set:
        return candidate

    norm_to_label = {
        normalize_celltype_label(x): x
        for x in available_labels
    }

    for option in [
        candidate,
        raw,
        raw.replace("_", " "),
        raw.replace("-", " "),
    ]:
        hit = norm_to_label.get(normalize_celltype_label(option))
        if hit is not None:
            return hit

    return None


def scpagwas_singlecell_file(base_dir: Path, trait: str) -> Path:
    """Return the scPagwas single-cell result file for a trait."""
    prefix = Path(trait).name
    base_dir = Path(base_dir)

    trait_dir = base_dir if base_dir.name == prefix else base_dir / prefix

    candidates = [
        trait_dir / f"{prefix}_snglecell_scPagwas_score_pvalue.Result.csv",
        trait_dir / f"{prefix}_singlecell_scPagwas_score_pvalue.Result.csv",
    ]

    for path in candidates:
        if path.exists():
            return path

    return candidates[0]


def scpagwas_celltype_file(base_dir: Path, trait: str) -> Path:
    """Return the scPagwas merged cell-type p-value file for a trait."""
    prefix = Path(trait).name
    base_dir = Path(base_dir)

    trait_dir = base_dir if base_dir.name == prefix else base_dir / prefix
    return trait_dir / f"{prefix}_Merged_celltype_pvalue.csv"


def build_scpagwas_tables(
    *,
    adata: sc.AnnData,
    base_dir: Path,
    traits: Sequence[str],
    biocol: str = "Subclass",
    cell_alpha: float = 0.1,
    ct_alpha: float = 0.05,
    score_sig_col: str = "Random_Correct_BG_adjp",
    ct_pval_col: str = "pvalue",
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """
    Build scPagwas heatmap inputs from both required files:

      1. Single-cell file:
           {trait}_snglecell_scPagwas_score_pvalue.Result.csv
         Significant cells are Random_Correct_BG_adjp < cell_alpha.

      2. Merged cell-type file:
           {trait}_Merged_celltype_pvalue.csv
         Associated cell types are BH-FDR significant at ct_alpha.

    Cell IDs are aligned with canonical SEA-AD IDs.
    scPagwas cell-type labels are mapped to adata.obs[biocol].
    """
    if biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs missing {biocol!r}")

    totals = celltype_counts(adata, biocol)
    celltype_order = totals.index.tolist()
    available_labels = adata.obs[biocol].astype(str).unique().tolist()

    prop_rows = {}
    assoc = {}

    for trait in traits:
        score_file = scpagwas_singlecell_file(base_dir, trait)
        ct_file = scpagwas_celltype_file(base_dir, trait)

        if not score_file.exists():
            print(f"Warning: missing scPagwas single-cell file for {trait}: {score_file}")
            prop_rows[trait] = pd.Series(0.0, index=celltype_order, name=trait)
            assoc[trait] = []
            continue

        if not ct_file.exists():
            print(f"Warning: missing scPagwas merged cell-type file for {trait}: {ct_file}")
            prop_rows[trait] = pd.Series(0.0, index=celltype_order, name=trait)
            assoc[trait] = []
            continue

        df_score = pd.read_csv(score_file, index_col=0)
        if score_sig_col not in df_score.columns:
            raise ValueError(f"{score_file} missing {score_sig_col!r}; columns={df_score.columns.tolist()}")

        aligned_score = align_external_cell_table_to_adata(
            df_score,
            adata,
            verbose=verbose,
        )

        sig_mask = pd.to_numeric(aligned_score[score_sig_col], errors="coerce") < cell_alpha
        sig_cells = pd.Index(aligned_score.index[sig_mask.fillna(False)].astype(str))

        prop_rows[trait] = discovery_fraction_by_celltype(
            adata,
            sig_cells,
            biocol=biocol,
            celltype_order=celltype_order,
            totals_by_type=totals,
        )

        df_ct = pd.read_csv(ct_file, index_col=0)
        if "celltype" not in df_ct.columns or ct_pval_col not in df_ct.columns:
            raise ValueError(f"{ct_file} must contain 'celltype' and {ct_pval_col!r}; columns={df_ct.columns.tolist()}")

        sig_ct = bh_fdr_mask(
            pd.to_numeric(df_ct[ct_pval_col], errors="coerce").to_numpy(),
            alpha=ct_alpha,
        )

        mapped = []
        for raw_celltype in df_ct.loc[sig_ct, "celltype"].astype(str):
            mapped_label = map_scpagwas_celltype_to_adata_label(
                raw_celltype,
                available_labels,
            )
            if mapped_label is None:
                print(f"Warning: could not map scPagwas celltype {raw_celltype!r} for {trait}")
            else:
                mapped.append(mapped_label)

        assoc[trait] = sorted(pd.Index(mapped).drop_duplicates().tolist())

    props = pd.DataFrame.from_dict(prop_rows, orient="index").astype(float)
    props.index.name = "trait"
    props.columns.name = biocol

    return props, assoc


def build_scdrs_props_and_region_supertype_assoc(
    *,
    adata: sc.AnnData,
    results_dir: Path,
    traits: Sequence[str],
    subclass_col: str = "Subclass",
    region_supertype_col: str = "Region_Supertype",
    fdr_alpha: float = 0.1,
    pval_col_candidates: Sequence[str] = ("pval", "mc_pval"),
    score_suffix: str = ".score.gz",
    region_supertype_threshold: float = 0.05,
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """
    Build scDRS heatmap inputs.

    Color values:
      - proportion of significant cells per Subclass.

    Stars:
      - a Subclass gets a star if any Region_Supertype belonging to that
        Subclass has > region_supertype_threshold significant cells.
    """
    for col in [subclass_col, region_supertype_col]:
        if col not in adata.obs.columns:
            raise ValueError(f"adata.obs missing {col!r}")

    props = build_marginal_props_from_score_files(
        adata=adata,
        results_dir=results_dir,
        traits=traits,
        biocol=subclass_col,
        fdr_alpha=fdr_alpha,
        pval_col_candidates=pval_col_candidates,
        score_suffix=score_suffix,
    )

    obs_names = adata.obs_names.astype(str)
    subclass = adata.obs[subclass_col].astype(str)
    region_supertype = adata.obs[region_supertype_col].astype(str)

    # Map each Region_Supertype to its dominant Subclass.
    region_to_subclass = (
        pd.DataFrame({
            "Region_Supertype": region_supertype.values,
            "Subclass": subclass.values,
        })
        .groupby("Region_Supertype")["Subclass"]
        .agg(lambda s: s.value_counts().index[0])
        .to_dict()
    )

    region_totals = region_supertype.value_counts()
    assoc = {}

    for trait in traits:
        prefix = Path(trait).name
        score_file = Path(results_dir) / f"{prefix}{score_suffix}"

        if not score_file.exists():
            print(f"Warning: missing scDRS score file for {trait}: {score_file}")
            assoc[trait] = []
            continue

        df = pd.read_csv(score_file, sep="\t", compression="infer", index_col=0)
        pcol = pick_first_existing_col(df, pval_col_candidates, what="scDRS p-value")

        sig_file = pd.Series(
            bh_fdr_mask(df[pcol].to_numpy(), alpha=fdr_alpha),
            index=df.index.astype(str),
        )

        sig_cells = obs_names.intersection(sig_file.index[sig_file].astype(str))

        if len(sig_cells) == 0:
            assoc[trait] = []
            continue

        sig_region_counts = region_supertype.loc[sig_cells].value_counts()
        frac_by_region = (
            sig_region_counts
            / region_totals.loc[sig_region_counts.index]
        ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

        sig_regions = frac_by_region.index[
            frac_by_region > region_supertype_threshold
        ].astype(str).tolist()

        assoc_subclasses = [
            region_to_subclass[r]
            for r in sig_regions
            if r in region_to_subclass
        ]

        assoc[trait] = sorted(pd.Index(assoc_subclasses).drop_duplicates().tolist())

    return props, assoc


def read_score_and_sig_from_scpagwas_file(
    file_path: Path,
    adata: sc.AnnData,
    *,
    score_col: str = "scPagwas.TRS.Score",
    sig_col: str = "Random_Correct_BG_adjp",
    sig_alpha: float = 0.1,
) -> tuple[pd.Series, pd.Series]:
    """
    Read a scPagwas single-cell file and align it to adata.obs_names
    using canonical SEA-AD cell IDs.
    """
    df = pd.read_csv(file_path, index_col=0)

    if score_col not in df.columns or sig_col not in df.columns:
        raise ValueError(
            f"{file_path} must contain {score_col!r} and {sig_col!r}; "
            f"columns={df.columns.tolist()}"
        )

    aligned = align_external_cell_table_to_adata(df, adata, verbose=True)

    scores = pd.to_numeric(aligned[score_col], errors="coerce")
    scores.index = adata.obs_names.astype(str)

    sig = pd.to_numeric(aligned[sig_col], errors="coerce") < sig_alpha
    sig = sig.fillna(False).astype(bool)
    sig.index = adata.obs_names.astype(str)

    return scores, sig


def plot_scdrsfm_ad_score_and_signals(
    adata_umap: sc.AnnData,
    *,
    score_key: str,
    signal_key: str,
    out_png: Path | str,
    score_title: str = "scDRS-FM AD disease scores",
    signal_title: str = "scDRS-FM independent populations",
    score_vmin: float | None = None,
    score_vmax: float | None = None,
    cellpop_key: str = "Subclass",
    base_size: float = 6,
    causal_size: float = 20,
    other_alpha: float = 0.3,
    cmap: str = "RdBu_r",
    center_at: float = 0.0,
    composition_min_ratio: float = 0.01,
    composition_min_cells: int = 50,
):
    """
    Two-panel SEA-AD AD UMAP.

    Left:
      conditional scDRS-FM AD disease scores.

    Highlighted cells:
      marginal × conditional significant cells, defined by signal_key >= 1.

    Right:
      independent populations, matching the T-cell notebook legend style.
    """
    if "X_umap" not in adata_umap.obsm:
        raise ValueError("adata_umap.obsm['X_umap'] not found. Run prepare_umap first.")

    for required in [score_key, signal_key, cellpop_key]:
        if required not in adata_umap.obs.columns:
            raise KeyError(f"{required!r} not found in adata_umap.obs")

    U = adata_umap.obsm["X_umap"]
    x, y = U[:, 0], U[:, 1]

    scores = pd.to_numeric(adata_umap.obs[score_key], errors="coerce").to_numpy()
    sig_raw = (
        pd.to_numeric(adata_umap.obs[signal_key].astype(str), errors="coerce")
        .fillna(-1)
        .astype(int)
        .to_numpy()
    )
    causal = sig_raw >= 1

    finite = np.isfinite(scores)
    if score_vmin is None or score_vmax is None:
        max_abs = float(np.nanquantile(np.abs(scores[finite]), 0.98)) if finite.any() else 1.0
        max_abs = max(max_abs, 1e-6)
        score_vmin = -max_abs if score_vmin is None else score_vmin
        score_vmax = max_abs if score_vmax is None else score_vmax

    norm = mcolors.TwoSlopeNorm(vcenter=center_at, vmin=score_vmin, vmax=score_vmax)

    cell_pops = adata_umap.obs[cellpop_key].astype(str).fillna("NA")
    total_by_celltype = cell_pops.value_counts()

    kept = sorted(np.unique(sig_raw[causal]).tolist())
    notsig_label = "Not sig."

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    # --------------------------------------------------------
    # Conditional disease-score panel.
    # --------------------------------------------------------
    ax = axes[0]
    ax.scatter(
        x[~causal],
        y[~causal],
        c=scores[~causal],
        cmap=cmap,
        norm=norm,
        s=base_size,
        alpha=other_alpha,
        linewidths=0,
        rasterized=True,
    )
    ax.scatter(
        x[causal],
        y[causal],
        c=scores[causal],
        cmap=cmap,
        norm=norm,
        s=causal_size,
        alpha=0.75,
        edgecolors="black",
        linewidths=0.4,
        rasterized=True,
    )
    ax.set_title(score_title, fontsize=18)

    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

    # --------------------------------------------------------
    # Independent-population panel.
    # --------------------------------------------------------
    ax2 = axes[1]
    notsig_color = "#D0D0D0"
    ax2.scatter(
        x[~causal],
        y[~causal],
        color=notsig_color,
        s=base_size,
        alpha=other_alpha,
        linewidths=0,
        rasterized=True,
    )

    if len(kept) <= 10:
        palette = sns.color_palette("tab10", n_colors=max(1, len(kept))).as_hex()
    elif len(kept) <= 20:
        palette = sns.color_palette("tab20", n_colors=len(kept)).as_hex()
    else:
        palette = sns.color_palette("husl", n_colors=len(kept)).as_hex()

    sig_to_color = {
        sig: palette[i]
        for i, sig in enumerate(kept)
    }

    handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markerfacecolor=notsig_color,
            markeredgecolor="black",  # legend-only outline
            markeredgewidth=0.8,
            markersize=7,
            alpha=other_alpha,
            label=notsig_label,
        )
    ]

    for sig in kept:
        mask = sig_raw == sig
        ax2.scatter(
            x[mask],
            y[mask],
            color=sig_to_color[sig],
            s=causal_size,
            alpha=0.75,
            edgecolors="black",
            linewidths=0.4,
            rasterized=True,
        )

        signal_cell_counts = cell_pops[mask].value_counts()
        comp_rows = []

        for ct, in_signal in signal_cell_counts.items():
            total_count = int(total_by_celltype.get(ct, 0))
            if total_count == 0:
                continue

            ratio = in_signal / total_count
            if ratio > composition_min_ratio and in_signal >= composition_min_cells:
                comp_rows.append((ct, int(in_signal), total_count, ratio))

        comp_rows = sorted(comp_rows, key=lambda z: (-z[3], -z[1], z[0]))

        if comp_rows:
            comp_text = "\n".join(
                f"   {ct}: {in_signal}/{total_count}"
                for ct, in_signal, total_count, _ in comp_rows
            )
            label = f"Indep. population {sig}\n{comp_text}"
        else:
            label = f"Indep. population {sig}"

        handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="None",
                markerfacecolor=sig_to_color[sig],
                markeredgecolor="black",
                markersize=8,
                label=label,
            )
        )

    ax2.set_title(signal_title, fontsize=18)
    legend = ax2.legend(
        handles=handles,
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=12,
        handletextpad=0.8,
        labelspacing=1.2,
        borderaxespad=0.0,
    )

    for text in legend.get_texts():
        text.set_fontsize(12)
        text.set_multialignment("left")
        text.set_fontfamily("monospace")

    for axis in axes:
        axis.set_xlabel("")
        axis.set_ylabel("")
        axis.set_xticks([])
        axis.set_yticks([])
        for spine in axis.spines.values():
            spine.set_visible(False)

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_png}")

## UMAP and disease-score plotting helpers

In [9]:

def prepare_umap(
    adata: sc.AnnData,
    *,
    n_neighbors: int = 15,
    n_pcs: int = 40,
    max_pcs: int = 50,
    force: bool = False,
) -> sc.AnnData:
    """Prepare or reuse a UMAP embedding."""
    adata_umap = adata.copy()

    if (not force) and "X_umap" in adata_umap.obsm:
        return adata_umap

    sc.pp.highly_variable_genes(
        adata_umap,
        subset=False,
        min_disp=0.5,
        min_mean=0.0125,
        max_mean=10,
        n_bins=20,
        n_top_genes=None,
    )
    sc.pp.scale(adata_umap, max_value=10, zero_center=False)

    n_comps = min(adata_umap.n_obs - 1, adata_umap.n_vars - 1, max_pcs)
    sc.pp.pca(adata_umap, n_comps=n_comps, use_highly_variable=True, svd_solver="arpack")
    sc.pp.neighbors(
        adata_umap,
        n_neighbors=min(n_neighbors, adata_umap.n_obs - 1),
        n_pcs=min(n_pcs, n_comps),
    )
    sc.tl.umap(adata_umap)
    return adata_umap


def clean_umap_axis(ax) -> None:
    """Remove ticks, labels, and frame from a UMAP axis."""
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def descending_score_indices(scores: Sequence[float], mask: Sequence[bool] | None = None) -> np.ndarray:
    """Return selected cell indices ordered by score from highest to lowest; NaNs are last."""
    values = np.asarray(scores, dtype=float)
    if mask is None:
        indices = np.arange(values.size)
    else:
        mask_arr = np.asarray(mask, dtype=bool)
        if mask_arr.shape != values.shape:
            raise ValueError("mask must have the same shape as scores")
        indices = np.flatnonzero(mask_arr)

    sortable = np.where(np.isfinite(values[indices]), values[indices], -np.inf)
    return indices[np.argsort(-sortable, kind="stable")]


def plot_categorical_umap_with_labels(
    adata_umap: sc.AnnData,
    *,
    color_key: str,
    title: str,
    out_png: Path | str,
    size: float = 6,
    label_fontsize: float = 8,
):
    """Categorical UMAP with on-data labels."""
    if color_key not in adata_umap.obs.columns:
        raise ValueError(f"adata.obs missing {color_key!r}")

    obs = adata_umap.obs[color_key].astype(str)
    categories = pd.Index(obs.unique())
    palette = sns.color_palette("husl", n_colors=len(categories)).as_hex()
    color_map = dict(zip(categories, palette))

    U = adata_umap.obsm["X_umap"]
    fig, ax = plt.subplots(figsize=(8, 8))

    for cat in categories:
        mask = obs.to_numpy() == cat
        ax.scatter(
            U[mask, 0],
            U[mask, 1],
            s=size,
            color=color_map[cat],
            linewidths=0,
            rasterized=True,
        )

    texts = []
    for cat in categories:
        mask = obs.to_numpy() == cat
        if mask.sum() == 0:
            continue
        txt = ax.text(
            np.median(U[mask, 0]),
            np.median(U[mask, 1]),
            f"● {cat}",
            color=color_map[cat],
            fontsize=label_fontsize,
            ha="center",
            va="center",
            bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.2", linewidth=0.8),
        )
        texts.append(txt)

    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="->", color="black", lw=1.0))
    ax.set_title(title, fontsize=24)
    clean_umap_axis(ax)

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {out_png}")


def explode_cell_ids(df: pd.DataFrame, *, cell_ids_col: str = "cell_ids") -> pd.DataFrame:
    """Explode comma-separated cell-id lists into long format."""
    out = df.copy()
    out["cell_id"] = out[cell_ids_col].astype(str).str.split(",")
    out = out.explode("cell_id", ignore_index=True)
    out["cell_id"] = out["cell_id"].astype(str).str.strip()
    out = out[(out["cell_id"] != "") & (out["cell_id"] != "nan")]
    return out


def assign_conditional_scores_to_cells_filtered_signals(
    *,
    adata_umap: sc.AnnData,
    results_dir: Path,
    trait: str,
    indep_sig_col: str = "independent_signal_multi",
    cell_ids_col: str = "cell_ids",
    score_col_candidates: Sequence[str] = ("tagging_score", "conditional_score", "score", "z", "zscore", "stat"),
    pval_col_candidates: Sequence[str] = ("pval", "mc_pval"),
    fdr_alpha: float = 0.1,
    marginal_metacell_col: str = "metacell",
    min_causal_cells_per_signal: int = 100,
    cellpop_key: str = "Subclass",
    min_fraction_within_any_cellpop: float = 0.01,
    obs_score_key: str | None = None,
    obs_sig_key: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[int, int]]:
    """
    Assign conditional scDRS-FM scores and filtered independent populations.

    Signal cells are explicitly marginal × conditional significant cells:
      marginally significant cells ∩ conditionally significant signal cells.
    """
    prefix = Path(trait).name
    df_cond = pd.read_csv(
        Path(results_dir) / f"{prefix}.conditional.tagging_score.gz",
        sep="\t",
        compression="infer",
        index_col=0,
    )
    df_marg = pd.read_csv(
        Path(results_dir) / f"{prefix}.marginal_score.gz",
        sep="\t",
        compression="infer",
        index_col=0,
    )

    score_col = pick_first_existing_col(df_cond, score_col_candidates, what="conditional score")
    pcol_cond = pick_first_existing_col(df_cond, pval_col_candidates, what="conditional p-value")
    pcol_marg = pick_first_existing_col(df_marg, pval_col_candidates, what="marginal p-value")

    obs_names = adata_umap.obs_names.astype(str)

    marginal_sig_cells = obs_names.intersection(
        df_marg.index[bh_fdr_mask(df_marg[pcol_marg].to_numpy(), alpha=fdr_alpha)].astype(str)
    )
    cond_sig_mask = bh_fdr_mask(df_cond[pcol_cond].to_numpy(), alpha=fdr_alpha)
    df_cond_sig = df_cond.loc[cond_sig_mask].copy()

    required_cols = [score_col, indep_sig_col, cell_ids_col]
    long_all = explode_cell_ids(df_cond[required_cols], cell_ids_col=cell_ids_col)
    long_all = long_all[long_all["cell_id"].isin(obs_names)]

    if len(df_cond_sig):
        long_sig = explode_cell_ids(df_cond_sig[required_cols], cell_ids_col=cell_ids_col)
    else:
        long_sig = long_all.iloc[0:0].copy()
    long_sig = long_sig[long_sig["cell_id"].isin(obs_names)]

    long_sig[indep_sig_col] = pd.to_numeric(long_sig[indep_sig_col], errors="coerce")
    long_sig = long_sig[long_sig[indep_sig_col].notna()]
    long_sig[indep_sig_col] = long_sig[indep_sig_col].astype(int)
    long_sig = long_sig[long_sig[indep_sig_col] >= 0]

    # Marginal × conditional cells by raw independent signal.
    causal_cells_by_signal: dict[int, pd.Index] = {}
    for sig, sub in long_sig.groupby(indep_sig_col, sort=True):
        causal_cells_by_signal[int(sig)] = marginal_sig_cells.intersection(
            pd.Index(sub["cell_id"].astype(str))
        )

    if cellpop_key not in adata_umap.obs.columns:
        raise ValueError(f"adata_umap.obs missing {cellpop_key!r}")

    cellpop = adata_umap.obs[cellpop_key].astype(str)
    cellpop.index = obs_names
    cellpop_totals = cellpop.value_counts()

    kept_old_signals: list[int] = []
    for sig, cells in causal_cells_by_signal.items():
        if len(cells) < min_causal_cells_per_signal:
            continue

        counts = cellpop.loc[cellpop.index.intersection(cells)].value_counts()
        if counts.empty:
            continue

        max_frac = float((counts / cellpop_totals.loc[counts.index]).max())
        if max_frac >= min_fraction_within_any_cellpop:
            kept_old_signals.append(sig)

    kept_old_signals = sorted(kept_old_signals)
    remap = {old: i + 1 for i, old in enumerate(kept_old_signals)}

    score_key = obs_score_key or f"{prefix}_conditional_score"
    sig_key = obs_sig_key or f"{prefix}_{indep_sig_col}_filtered"

    # Conditional scores for all cells represented in the conditional tagging table.
    score_s = pd.Series(np.nan, index=obs_names, dtype=float)
    if len(long_all):
        tmp = long_all[["cell_id", score_col]].copy()
        tmp[score_col] = pd.to_numeric(tmp[score_col], errors="coerce")
        # If a cell appears more than once, keep the largest absolute conditional score.
        tmp["abs_score"] = tmp[score_col].abs()
        tmp = tmp.sort_values("abs_score").drop_duplicates("cell_id", keep="last")
        score_s.loc[tmp["cell_id"].astype(str).values] = tmp[score_col].to_numpy(dtype=float)

    sig_s = pd.Series(-1, index=obs_names, dtype=int)
    for old_sig in kept_old_signals:
        cells = obs_names.intersection(causal_cells_by_signal.get(old_sig, pd.Index([], dtype=str)))
        sig_s.loc[cells] = remap[old_sig]

    adata_umap.obs[score_key] = score_s.reindex(adata_umap.obs_names).to_numpy()
    adata_umap.obs[sig_key] = pd.Categorical(
        sig_s.reindex(adata_umap.obs_names).astype(int),
        categories=[-1] + list(remap.values()),
        ordered=True,
    )

    return df_cond, df_marg, remap


def plot_scdrsfm_ad_score_and_signals(
    adata_umap: sc.AnnData,
    *,
    score_key: str,
    signal_key: str,
    out_png: Path | str,
    cellpop_key: str = "Subclass",
    score_title: str = "scDRS-FM disease scores (AD)",
    signal_title: str = "scDRS-FM independent populations (AD)",
    notsig_label: str = "Not sig.",
    base_size: float = 6,
    causal_size: float = 20,
    signal_size: float = 14,
    other_alpha: float = 0.3,
    cmap: str = "RdBu_r",
    center_at: float = 0.0,
    composition_min_ratio: float = 0.01,
    composition_min_cells: int = 50,
    dpi: int = 300,
    show: bool = True,
):
    """
    Two-panel trait UMAP matching the T-cell notebook style.

    Left panel:
      conditional scDRS-FM scores, with marginal × conditional cells emphasized.
      Cells are drawn in descending disease-score order within each significance layer.

    Right panel:
      independent populations, using the T-cell legend format. Significant
      population points use ``signal_size`` so they can be smaller than the
      highlighted points in the score panel. The gray legend marker has a
      black outline, but the plot's gray background points remain unoutlined.
    """
    if "X_umap" not in adata_umap.obsm:
        raise ValueError("adata_umap.obsm['X_umap'] not found. Run prepare_umap first.")

    for required in [score_key, signal_key, cellpop_key]:
        if required not in adata_umap.obs:
            raise KeyError(f"{required!r} not found in adata_umap.obs")

    U = adata_umap.obsm["X_umap"]
    x, y = U[:, 0], U[:, 1]

    scores = pd.to_numeric(adata_umap.obs[score_key], errors="coerce").to_numpy()
    sig_raw = pd.to_numeric(adata_umap.obs[signal_key].astype(str), errors="coerce").fillna(-1).astype(int).to_numpy()
    causal = sig_raw != -1
    background_order = descending_score_indices(scores, ~causal)
    causal_order = descending_score_indices(scores, causal)

    cell_pops = adata_umap.obs[cellpop_key].astype(str).fillna("NA")
    total_by_celltype = cell_pops.value_counts()
    kept = sorted(np.unique(sig_raw[causal]).tolist())

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    finite = np.isfinite(scores)
    max_abs = 5#float(np.nanmax(np.abs(scores[finite] - center_at))) if finite.any() else 1.0
    if max_abs == 0:
        max_abs = 1.0
    norm = mcolors.TwoSlopeNorm(vcenter=center_at, vmin=center_at - max_abs, vmax=center_at + max_abs)

    ax = axes[0]
    ax.scatter(
        x[background_order],
        y[background_order],
        c=scores[background_order],
        cmap=cmap,
        norm=norm,
        s=base_size,
        alpha=other_alpha,
        linewidths=0,
        rasterized=True,
    )
    ax.scatter(
        x[causal_order],
        y[causal_order],
        c=scores[causal_order],
        cmap=cmap,
        norm=norm,
        s=causal_size,
        alpha=0.65,
        edgecolors="black",
        linewidths=0.4,
        rasterized=True,
    )
    ax.set_title(score_title, fontsize=18)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

    ax2 = axes[1]
    notsig_color = "#D0D0D0"
    ax2.scatter(
        x[background_order],
        y[background_order],
        color=notsig_color,
        s=base_size,
        alpha=other_alpha,
        linewidths=0,
        rasterized=True,
    )

    if len(kept) <= 10:
        palette = sns.color_palette("tab10", n_colors=max(1, len(kept))).as_hex()
    elif len(kept) <= 20:
        palette = sns.color_palette("tab20", n_colors=len(kept)).as_hex()
    else:
        palette = sns.color_palette("husl", n_colors=len(kept)).as_hex()
    sig_to_color = {s: palette[i] for i, s in enumerate(kept)}

    handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markerfacecolor=notsig_color,
            markeredgecolor="black",  # legend-only outline; plot dots remain unoutlined
            markeredgewidth=0.8,
            markersize=7,
            alpha=other_alpha,
            label=notsig_label,
        )
    ]

    for s in kept:
        mask = sig_raw == s
        signal_order = descending_score_indices(scores, mask)
        ax2.scatter(
            x[signal_order],
            y[signal_order],
            color=sig_to_color[s],
            s=signal_size,
            alpha=0.65,
            edgecolors="black",
            linewidths=0.4,
            rasterized=True,
        )

        signal_cell_counts = cell_pops[mask].value_counts()
        comp_rows = []
        for ct, in_signal in signal_cell_counts.items():
            total_count = int(total_by_celltype.get(ct, 0))
            if total_count == 0:
                continue
            ratio = in_signal / total_count
            if ratio > composition_min_ratio and in_signal >= composition_min_cells:
                comp_rows.append((ct, int(in_signal), total_count, ratio))
        comp_rows = sorted(comp_rows, key=lambda z: (-z[3], -z[1], z[0]))

        if comp_rows:
            comp_text = "\n".join(
                f"   {ct}: {in_signal}/{total_count}"
                for ct, in_signal, total_count, _ in comp_rows
            )
            label = f"Indep. population {s}\n{comp_text}"
        else:
            label = f"Indep. population {s}"

        handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="None",
                markerfacecolor=sig_to_color[s],
                markeredgecolor="black",
                markersize=8,
                label=label,
            )
        )

    ax2.set_title(signal_title, fontsize=18)
    legend = ax2.legend(
        handles=handles,
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=12,
        handletextpad=0.8,
        labelspacing=1.2,
        borderaxespad=0.0,
    )
    for text in legend.get_texts():
        text.set_fontsize(12)
        text.set_multialignment("left")
        text.set_fontfamily("monospace")

    for axis in axes:
        clean_umap_axis(axis)

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=dpi, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    print(f"Saved {out_png}")


def _normalize_cell_id_for_alignment(x: object) -> str:
    """Conservative cell-id normalization for cross-tool score alignment."""
    return str(x).strip().replace("_", "-")


def _aligned_series_from_frame(
    df: pd.DataFrame,
    adata: sc.AnnData,
    column: str,
    *,
    dtype=float,
) -> pd.Series:
    """
    Align a dataframe column to adata.obs_names.

    Direct matches are used first. If needed, a conservative normalized-id
    match is attempted for unmatched cells.
    """
    if column not in df.columns:
        raise ValueError(f"Input dataframe missing {column!r}; columns={df.columns.tolist()}")

    obs = pd.Index(adata.obs_names.astype(str))
    df = df.copy()
    df.index = df.index.astype(str)

    out = pd.Series(np.nan if dtype is float else False, index=obs)

    common = obs.intersection(df.index)
    if len(common):
        out.loc[common] = df.loc[common, column].to_numpy()

    # Fallback normalized matching when direct overlap is poor.
    if len(common) < max(10, 0.05 * len(obs)):
        obs_norm = pd.Series(obs, index=[_normalize_cell_id_for_alignment(x) for x in obs])
        obs_norm = obs_norm[~obs_norm.index.duplicated(keep=False)]

        df_norm = pd.Series(df.index, index=[_normalize_cell_id_for_alignment(x) for x in df.index])
        df_norm = df_norm[~df_norm.index.duplicated(keep=False)]

        shared_norm = obs_norm.index.intersection(df_norm.index)
        if len(shared_norm):
            obs_ids = obs_norm.loc[shared_norm].to_numpy()
            df_ids = df_norm.loc[shared_norm].to_numpy()
            out.loc[obs_ids] = df.loc[df_ids, column].to_numpy()

    return out


def read_score_and_sig_from_scdrs_like_file(
    file_path: Path,
    adata: sc.AnnData,
    *,
    score_col_candidates=("norm_score", "score", "scdrs_score", "zscore"),
    pval_col_candidates=("pval", "mc_pval"),
    fdr_alpha=0.1,
) -> tuple[pd.Series, pd.Series]:
    """Read a scDRS/scDRS-FM score file and return aligned scores plus BH-FDR significant cells."""
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(file_path)

    df = pd.read_csv(file_path, sep="\t", compression="infer", index_col=0)
    score_col = pick_first_existing_col(df, score_col_candidates, what=f"{file_path} score")
    pcol = pick_first_existing_col(df, pval_col_candidates, what=f"{file_path} p-value")

    scores = _aligned_series_from_frame(df, adata, score_col)
    scores = pd.to_numeric(scores, errors="coerce")

    sig_file = pd.Series(bh_fdr_mask(df[pcol].to_numpy(), alpha=fdr_alpha), index=df.index.astype(str), name="sig")
    sig_df = sig_file.to_frame()
    sig = _aligned_series_from_frame(sig_df, adata, "sig", dtype=bool).fillna(False).astype(bool)

    print(f"{file_path.name}: aligned {scores.notna().sum():,} / {adata.n_obs:,} cells")
    return scores, sig


def read_score_and_sig_from_scpagwas_file(
    file_path: Path,
    adata: sc.AnnData,
    *,
    score_col: str = "scPagwas.TRS.Score",
    sig_col: str = "Random_Correct_BG_adjp",
    sig_alpha: float = 0.1,
) -> tuple[pd.Series, pd.Series]:
    """Read a scPagwas single-cell result file and return aligned scores/significant-cell mask."""
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(file_path)

    df = pd.read_csv(file_path, index_col=0)
    if score_col not in df.columns or sig_col not in df.columns:
        raise ValueError(
            f"{file_path} must contain {score_col!r} and {sig_col!r}; "
            f"columns={df.columns.tolist()}"
        )

    scores = _aligned_series_from_frame(df, adata, score_col)
    scores = pd.to_numeric(scores, errors="coerce")

    sig_df = (pd.to_numeric(df[sig_col], errors="coerce") < sig_alpha).rename("sig").to_frame()
    sig = _aligned_series_from_frame(sig_df, adata, "sig", dtype=bool).fillna(False).astype(bool)

    print(f"{file_path.name}: aligned {scores.notna().sum():,} / {adata.n_obs:,} cells")
    return scores, sig


def plot_score_umap_comparison(
    adata_umap: sc.AnnData,
    panels: Sequence[tuple[str, pd.Series, pd.Series]],
    *,
    out_png: Path | str,
    cmap: str = "RdBu_r",
    base_size: float = 5,
    sig_size: float = 24,
    base_alpha: float = 0.35,
    sig_alpha: float = 0.95,
    show: bool = True,
):
    """Multi-panel disease-score UMAP with significant cells emphasized."""
    U = adata_umap.obsm["X_umap"]
    fig, axes = plt.subplots(1, len(panels), figsize=(5.2 * len(panels), 5.2), constrained_layout=True)
    axes = np.ravel(axes)

    for ax, (title, scores, sig) in zip(axes, panels):
        scores = pd.to_numeric(scores.reindex(adata_umap.obs_names), errors="coerce")
        sig = sig.reindex(adata_umap.obs_names).fillna(False).astype(bool)

        vals = scores.to_numpy(dtype=float)
        finite = np.isfinite(vals)
        max_abs = np.nanquantile(np.abs(vals[finite]), 0.98) if finite.any() else 1.0
        max_abs = max(float(max_abs), 1e-6)
        norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)

        sig_arr = sig.to_numpy()
        background_order = descending_score_indices(vals, ~sig_arr)
        sig_order = descending_score_indices(vals, sig_arr)

        ax.scatter(
            U[background_order, 0],
            U[background_order, 1],
            c=vals[background_order],
            cmap=cmap,
            norm=norm,
            s=base_size,
            linewidths=0,
            alpha=base_alpha,
            rasterized=True,
        )
        ax.scatter(
            U[sig_order, 0],
            U[sig_order, 1],
            c=vals[sig_order],
            cmap=cmap,
            norm=norm,
            s=sig_size,
            edgecolors="black",
            linewidths=0.5,
            alpha=sig_alpha,
            rasterized=True,
        )
        ax.set_title(title, fontsize=16)
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
        clean_umap_axis(ax)

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    print(f"Saved {out_png}")


def plot_score_file_umap(
    adata_umap: sc.AnnData,
    score_file: Path,
    *,
    score_col: str = "norm_score",
    multiply_by: float = 1.0,
    title: str,
    out_png: Path | str,
    cell_type: str | None = None,
    show: bool = True,
):
    """Plot a single score file as a UMAP without highlighted cells.

    If cell_type is provided, subset to adata_umap.obs["Subclass"] == cell_type.
    """
    score_file = Path(score_file)
    df = pd.read_csv(score_file, sep="\t", compression="infer", index_col=0)

    if score_col not in df.columns:
        raise ValueError(f"{score_file} missing {score_col!r}; columns={df.columns.tolist()}")

    if cell_type is not None:
        if "Subclass" not in adata_umap.obs.columns:
            raise ValueError("adata_umap.obs is missing required column 'Subclass'")

        mask = adata_umap.obs["Subclass"] == cell_type
        if not mask.any():
            raise ValueError(f"No cells found with adata_umap.obs['Subclass'] == {cell_type!r}")

        adata_plot = adata_umap[mask].copy()
    else:
        adata_plot = adata_umap

    scores = _aligned_series_from_frame(df, adata_plot, score_col)
    scores = pd.to_numeric(scores, errors="coerce") * multiply_by

    sig = pd.Series(False, index=adata_plot.obs_names)

    plot_score_umap_comparison(
        adata_plot,
        [(title, scores, sig)],
        out_png=out_png,
        base_alpha=0.9,
        show=show,
    )

    return scores

## Load SEA-AD data

In [10]:
adata = load_and_preprocess_adata(SEA_AD_H5AD, normalize=False)
ensure_sea_ad_obs_columns(adata)

print("Subclass labels:", adata.obs["Subclass"].nunique())
print("Region_Supertype labels:", adata.obs["Region_Supertype"].nunique())
print("Region_Subclass labels:", adata.obs["Region_Subclass"].nunique())

Loaded /mnt/shared-workspace/scdrsfm/data/subsets_10k/SEA_AD/combined_healthy_filtered.h5ad: 10,000 cells × 36,601 genes


After filtering: 10,000 cells × 23,685 genes
Subclass labels: 24
Region_Supertype labels: 369
Region_Subclass labels: 77


## Build SEA-AD scDRS-FM tables and export independent signals

In [11]:
sea_subclass_tables = build_scdrsfm_celltype_tables(
    adata=adata,
    results_dir=SEA_AD_SCDRSFM_DIR,
    traits=SCDRSFM_TRAITS,
    biocol="Subclass",
    marginal_metacell_col="metacell",
    fdr_alpha=FDR_ALPHA,
    pval_col_candidates=("pval",),
    indep_sig_col="independent_signal_multi",
    indep_cells_dir=INDEP_CELLS_DIR,
    heatmap_threshold=HEATMAP_THRESHOLD,
    heatmap_traits=SCDRSFM_TRAITS,
    print_summaries=True,
)

sea_region_supertype_tables = build_scdrsfm_celltype_tables(
    adata=adata,
    results_dir=SEA_AD_SCDRSFM_DIR,
    traits=SCDRSFM_TRAITS,
    biocol="Region_Supertype",
    marginal_metacell_col="metacell",
    fdr_alpha=FDR_ALPHA,
    pval_col_candidates=("pval",),
    indep_sig_col="independent_signal_multi",
    indep_cells_dir=None,
    print_summaries=False,
)

display(sea_subclass_tables["df_indep_signal_counts"])
print(f"Independent cell assignments written to {INDEP_CELLS_DIR}")

PASS_Parkinsons23andMe_Corces2020: 0 marginal cells; 0 marginal∩conditional cells; indep assignments raw=0, saved in sections >1.0%=0


PASS_Alzheimers_Jansen2019: 891 marginal cells; 268 marginal∩conditional cells; indep assignments raw=268, saved in sections >1.0%=268


PASS_ADHD_Demontis2018: 0 marginal cells; 0 marginal∩conditional cells; indep assignments raw=0, saved in sections >1.0%=0


PASS_BIP_Mullins2021: 2,181 marginal cells; 214 marginal∩conditional cells; indep assignments raw=214, saved in sections >1.0%=210


PASS_Intelligence_SavageJansen2018: 2,355 marginal cells; 427 marginal∩conditional cells; indep assignments raw=427, saved in sections >1.0%=422


PASS_Schizophrenia_Pardinas2018: 1,566 marginal cells; 6 marginal∩conditional cells; indep assignments raw=6, saved in sections >1.0%=0


PASS_MDD_Howard2019: 3,441 marginal cells; 200 marginal∩conditional cells; indep assignments raw=200, saved in sections >1.0%=192


UKB_460K.mental_NEUROTICISM: 3,822 marginal cells; 526 marginal∩conditional cells; indep assignments raw=526, saved in sections >1.0%=518


,n_independent_signals_total,n_independent_signals_cond_sig,n_independent_signals_with_any_causal_cells,n_marginal_sig_cells,n_cond_sig_cells,n_causal_cells_marg_x_cond,n_indep_cell_assignments_before_heatmap_filter,n_indep_cell_assignments_saved_after_heatmap_filter
trait,,,,,,,,
PASS_Parkinsons23andMe_Corces2020,3,3,0,0,100,0,0,0
PASS_Alzheimers_Jansen2019,15,4,3,891,526,268,268,268
PASS_ADHD_Demontis2018,2,2,0,0,131,0,0,0
PASS_BIP_Mullins2021,36,8,7,2181,509,214,214,210
PASS_Intelligence_SavageJansen2018,2,1,1,2355,732,427,427,422
PASS_Schizophrenia_Pardinas2018,6,1,1,1566,8,6,6,0
PASS_MDD_Howard2019,5,3,2,3441,467,200,200,192
UKB_460K.mental_NEUROTICISM,7,5,4,3822,816,526,526,518


Independent cell assignments written to indep_cells/sea_ad


## SEA-AD scDRS-FM heatmap: Subclass

This plot displays subclass-level conditional proportions and keeps any subclass with at least one marginal association.

In [12]:
sea_main_celltypes = plot_scdrsfm_heatmap(
    df_marginal_props=sea_subclass_tables["df_marginal_props"],
    df_intersect_props=sea_subclass_tables["df_marginal_x_cond_props"],
    adata=adata,
    biocol="Subclass",
    trait_order=SCDRSFM_TRAITS,
    trait_labels=TRAIT_LABELS,
    threshold=HEATMAP_THRESHOLD,
    out_png=OUTPUT_DIR / "sea_ad_scdrsfm_subclass_heatmap.png",
    out_csv=SEA_AD_SCDRSFM_SUBCLASS_CSV,
    title="",
    df_signal_details=sea_subclass_tables["df_signal_details"],
    signal_cell_ids_col="marg_x_signal_cell_ids_cond_sig",
    celltype_order=SEA_SUBCLASS_ORDER,
    celltype_groups=SEA_SUBCLASS_GROUPS,
    celltype_group_colors=SEA_GROUP_COLORS,
    annotation_biocol="Subclass",
    annotation_threshold=HEATMAP_THRESHOLD,
    fontsize_mult=1.35,
    cbar_label="Prop. sig. conditional cells",
)
sea_main_celltypes

Saved heatmap cell-type proportions: nature_genetics_manuscript_supplementary/SEA_AD_scDRSFM_subclass_heatmap_cell_type_proportions.csv (168 rows)
Saved: sea_ad_analysis_outputs/sea_ad_scdrsfm_subclass_heatmap.png


['Astrocyte',
 'Microglia-PVM',
 'OPC',
 'L2/3 IT',
 'L4 IT',
 'L5 ET',
 'L5 IT',
 'L5/6 NP',
 'L6 CT',
 'L6 IT',
 'L6 IT Car3',
 'L6b',
 'Chandelier',
 'Lamp5',
 'Lamp5 Lhx6',
 'Pax6',
 'Pvalb',
 'Sncg',
 'Sst',
 'Sst Chodl',
 'Vip']

## SEA-AD alternative heatmap: stars from Region_Supertype, values from Subclass

The color scale still displays total subclass-level conditional proportions. Hollow/red stars are based on whether any `Region_Supertype` label belonging to that subclass exceeds 5% for marginal/conditional association.

In [13]:

fine_to_subclass = fine_to_parent_map_from_adata(
    adata,
    fine_col="Region_Supertype",
    parent_col="Subclass",
)

subclass_to_region_supertypes: dict[str, list[str]] = {}
for fine_label, subclass in fine_to_subclass.items():
    subclass_to_region_supertypes.setdefault(subclass, []).append(fine_label)

# Region_Supertype order/grouping, used by validation heatmaps below.
REGION_SUPERTYPE_ORDER = []
for subclass in SEA_SUBCLASS_ORDER:
    REGION_SUPERTYPE_ORDER.extend(sorted(subclass_to_region_supertypes.get(subclass, [])))
REGION_SUPERTYPE_ORDER.extend(
    sorted(
        fine_label
        for fine_label in fine_to_subclass
        if fine_label not in set(REGION_SUPERTYPE_ORDER)
    )
)

REGION_SUPERTYPE_GROUPS = {}
for group, subclasses in SEA_SUBCLASS_GROUPS.items():
    members = []
    for subclass in subclasses:
        members.extend(subclass_to_region_supertypes.get(subclass, []))
    REGION_SUPERTYPE_GROUPS[group] = sorted(members)

region_star_marg = collapse_fine_props_to_parent_max(
    sea_region_supertype_tables["df_marginal_props"],
    fine_to_parent=fine_to_subclass,
    parent_order=sea_subclass_tables["df_marginal_props"].columns,
)
region_star_cond = collapse_fine_props_to_parent_max(
    sea_region_supertype_tables["df_marginal_x_cond_props"],
    fine_to_parent=fine_to_subclass,
    parent_order=sea_subclass_tables["df_marginal_x_cond_props"].columns,
)

sea_region_rule_celltypes = plot_scdrsfm_heatmap(
    df_marginal_props=sea_subclass_tables["df_marginal_props"],
    df_intersect_props=sea_subclass_tables["df_marginal_x_cond_props"],
    adata=adata,
    biocol="Subclass",
    trait_order=SCDRSFM_TRAITS,
    trait_labels=TRAIT_LABELS,
    threshold=HEATMAP_THRESHOLD,
    out_png=OUTPUT_DIR / "sea_ad_scdrsfm_subclass_heatmap_region_supertype_stars.png",
    out_csv=SEA_AD_SCDRSFM_REGION_STAR_CSV,
    title="",
    df_signal_details=sea_subclass_tables["df_signal_details"],
    signal_cell_ids_col="marg_x_signal_cell_ids_cond_sig",
    celltype_order=SEA_SUBCLASS_ORDER,
    celltype_groups=SEA_SUBCLASS_GROUPS,
    celltype_group_colors=SEA_GROUP_COLORS,
    star_marginal_props=region_star_marg,
    star_conditional_props=region_star_cond,
    star_threshold=REGION_SUPERTYPE_STAR_THRESHOLD,
    annotation_biocol="Region_Supertype",
    annotation_threshold=REGION_SUPERTYPE_STAR_THRESHOLD,
    display_to_annotation_labels=subclass_to_region_supertypes,
    fontsize_mult=1.35,
    cbar_label="Subclass prop. sig. conditional cells",
)

sea_region_rule_celltypes


Saved heatmap cell-type proportions: nature_genetics_manuscript_supplementary/SEA_AD_scDRSFM_subclass_heatmap_region_supertype_stars_cell_type_proportions.csv (168 rows)
Saved: sea_ad_analysis_outputs/sea_ad_scdrsfm_subclass_heatmap_region_supertype_stars.png


['Astrocyte',
 'Microglia-PVM',
 'OPC',
 'L2/3 IT',
 'L4 IT',
 'L5 ET',
 'L5 IT',
 'L5/6 NP',
 'L6 CT',
 'L6 IT',
 'L6 IT Car3',
 'L6b',
 'Chandelier',
 'Lamp5',
 'Lamp5 Lhx6',
 'Pax6',
 'Pvalb',
 'Sncg',
 'Sst',
 'Sst Chodl',
 'Vip']

## SEA-AD scDRS and scPagwas all-trait heatmaps

These validation heatmaps use the comparison-trait subset defined below. The scDRS-FM UMAP section retains the complete `SCDRSFM_TRAITS` list, while the cross-method UMAP comparisons use the traits available to all three methods.

In [14]:
SUBSET_TRAITS = [ 'PASS_Alzheimers_Jansen2019',
 'PASS_ADHD_Demontis2018',
 'PASS_BIP_Mullins2021',
 'PASS_Intelligence_SavageJansen2018',
 'PASS_Schizophrenia_Pardinas2018',
 'PASS_MDD_Howard2019',
 'UKB_460K.mental_NEUROTICISM']

In [15]:
# === scPagwas guard (injected, P5): skip scPagwas cleanly when its data is absent ===
# User decision (2026-07-18): guard scPagwas reader fns to return empty + auto-skip
# the scPagwas-only plots.  scDRS / scDRS-FM logic and figures are untouched.
import functools as _functools

_SCPAGWAS_PRESENT = bool(SEA_AD_SCPAGWAS_ROOT) and _Path(SEA_AD_SCPAGWAS_ROOT).exists()
print(f"[guard] scPagwas data present: {_SCPAGWAS_PRESENT} ({SEA_AD_SCPAGWAS_ROOT})")


def _empty_sig_pair(adata_or_cells):
    import numpy as _np
    cells = getattr(adata_or_cells, "obs_names", adata_or_cells)
    s = pd.Series(_np.nan, index=pd.Index(cells, dtype=str), dtype=float)
    m = pd.Series(False, index=pd.Index(cells, dtype=str), dtype=bool)
    s.attrs["scpagwas_absent"] = True
    return s, m


if not _SCPAGWAS_PRESENT:
    # 1. reader used inside the per-trait UMAP loop -> empty (blank panel, no crash)
    def read_score_and_sig_from_scpagwas_file(score_file, adata, *, score_col="scPagwas.TRS.Score",
                                              sig_col="Random_Correct_BG_adjp", sig_alpha=0.1, **kw):
        return _empty_sig_pair(adata)

    # 2. table builder used by the scPagwas heatmap cell -> empty, sentinel-tagged
    def build_scpagwas_tables(*, adata, base_dir, traits, biocol="Subclass", cell_alpha=0.1,
                              ct_alpha=0.05, score_sig_col="Random_Correct_BG_adjp",
                              ct_pval_col="pvalue", verbose=False, **kw):
        df_props = pd.DataFrame(index=pd.Index([], name="trait"))
        df_props.attrs["scpagwas_absent"] = True
        df_assoc = pd.DataFrame(index=pd.Index([], name="trait"))
        df_assoc.attrs["scpagwas_absent"] = True
        return df_props, df_assoc

    # 3. shared heatmap plotter -> early no-op ONLY for empty/sentinel scPagwas data
    _orig_plot_marg = plot_marginal_method_heatmap

    @_functools.wraps(_orig_plot_marg)
    def plot_marginal_method_heatmap(*args, **kwargs):
        prop = kwargs.get("df_props")
        if prop is None and args:
            prop = args[0]
        if isinstance(prop, pd.DataFrame) and (prop.attrs.get("scpagwas_absent") or prop.empty):
            print("[guard] scPagwas heatmap skipped (no scPagwas data).")
            return []
        return _orig_plot_marg(*args, **kwargs)


[guard] scPagwas data present: False (/mnt/shared-workspace/scdrsfm/results/scpagwas/sea_ad)


In [16]:
scdrs_props, scdrs_region_supertype_assoc = build_scdrs_props_and_region_supertype_assoc(
    adata=adata,
    results_dir=SEA_AD_SCDRS_DIR,
    traits=SUBSET_TRAITS,
    subclass_col="Subclass",
    region_supertype_col="Subclass",
    fdr_alpha=0.1,
    pval_col_candidates=("pval", "mc_pval"),
    score_suffix=".marginal_score.gz",
    region_supertype_threshold=0.05,
)

plot_marginal_method_heatmap(
    df_props=scdrs_props,
    associated_celltypes=scdrs_region_supertype_assoc,
    adata=adata,
    biocol="Subclass",
    trait_order=SUBSET_TRAITS,
    trait_labels=TRAIT_LABELS,
    out_png=OUTPUT_DIR / "sea_ad_scdrs_all_traits_region_supertype_assoc_heatmap.png",
    out_csv=SEA_AD_SCDRS_CSV,
    title="scDRS cell-type associations",
    threshold=HEATMAP_THRESHOLD,
    celltype_order=SEA_SUBCLASS_ORDER,
    celltype_groups=SEA_SUBCLASS_GROUPS,
    celltype_group_colors=SEA_GROUP_COLORS,
    cbar_label="Prop. sig. scDRS cells",
)



scpagwas_props, scpagwas_assoc = build_scpagwas_tables(
    adata=adata,
    base_dir=SEA_AD_SCPAGWAS_ROOT,
    traits=SUBSET_TRAITS,
    biocol="Subclass",
    cell_alpha=0.1,
    ct_alpha=0.05,
    score_sig_col="Random_Correct_BG_adjp",
    ct_pval_col="pvalue",
    verbose=True,
)

plot_marginal_method_heatmap(
    df_props=scpagwas_props,
    associated_celltypes=scpagwas_assoc,
    adata=adata,
    biocol="Subclass",
    trait_order=SUBSET_TRAITS,
    trait_labels=TRAIT_LABELS,
    out_png=OUTPUT_DIR / "sea_ad_scpagwas_all_traits_heatmap.png",
    out_csv=SEA_AD_SCPAGWAS_CSV,
    title="scPagwas cell-type associations",
    threshold=HEATMAP_THRESHOLD,
    celltype_order=SEA_SUBCLASS_ORDER,
    celltype_groups=SEA_SUBCLASS_GROUPS,
    celltype_group_colors=SEA_GROUP_COLORS,
    cbar_label="Prop. sig. scPagwas cells",
)



Saved heatmap cell-type proportions: nature_genetics_manuscript_supplementary/SEA_AD_scDRS_all_traits_heatmap_cell_type_proportions.csv (112 rows)
Saved: sea_ad_analysis_outputs/sea_ad_scdrs_all_traits_region_supertype_assoc_heatmap.png
[guard] scPagwas heatmap skipped (no scPagwas data).


[]

## Braun / Linnarsson scDRS-FM heatmaps

In [17]:
braun_adata = load_and_preprocess_adata(BRAUN_H5AD, normalize=False)
if "CellClass" not in braun_adata.obs.columns:
    raise ValueError("Braun AnnData must contain adata.obs['CellClass']")
if "Region" not in braun_adata.obs.columns:
    raise ValueError("Braun AnnData must contain adata.obs['Region']")
braun_adata.obs["CellClass"] = braun_adata.obs["CellClass"].astype(str)
braun_adata.obs["Region"] = braun_adata.obs["Region"].astype(str)
braun_adata.obs["Region_CellClass"] = braun_adata.obs["Region"] + "_" + braun_adata.obs["CellClass"]

braun_tables = build_scdrsfm_celltype_tables(
    adata=braun_adata,
    results_dir=BRAUN_SCDRSFM_DIR,
    traits=SUBSET_TRAITS,
    biocol="CellClass",
    marginal_metacell_col="metacell",
    fdr_alpha=FDR_ALPHA,
    pval_col_candidates=("pval",),
    indep_sig_col="independent_signal",
    indep_cells_dir=None,
    print_summaries=True,
)

braun_region_tables = build_scdrsfm_celltype_tables(
    adata=braun_adata,
    results_dir=BRAUN_SCDRSFM_DIR,
    traits=SUBSET_TRAITS,
    biocol="Region_CellClass",
    marginal_metacell_col="metacell",
    fdr_alpha=FDR_ALPHA,
    pval_col_candidates=("pval",),
    indep_sig_col="independent_signal",
    indep_cells_dir=None,
    print_summaries=False,
)

braun_cellclass_order = braun_adata.obs["CellClass"].astype(str).value_counts().index.tolist()
braun_cellclass_groups = {"CellClass": braun_cellclass_order}
braun_cellclass_colors = {"CellClass": "black"}

plot_scdrsfm_heatmap(
    df_marginal_props=braun_tables["df_marginal_props"],
    df_intersect_props=braun_tables["df_marginal_x_cond_props"],
    adata=braun_adata,
    biocol="CellClass",
    trait_order=SUBSET_TRAITS,
    trait_labels=TRAIT_LABELS,
    threshold=HEATMAP_THRESHOLD,
    out_png=OUTPUT_DIR / "braun_scdrsfm_cellclass_heatmap.png",
    out_csv=BRAUN_SCDRSFM_CELLCLASS_CSV,
    title="",
    df_signal_details=braun_tables["df_signal_details"],
    celltype_order=braun_cellclass_order,
    celltype_groups=braun_cellclass_groups,
    celltype_group_colors=braun_cellclass_colors,
    annotation_biocol="CellClass",
    annotation_threshold=HEATMAP_THRESHOLD,
    fontsize_mult=1.20,
)

braun_fine_to_class = fine_to_parent_map_from_adata(braun_adata, fine_col="Region_CellClass", parent_col="CellClass")
braun_class_to_region = {}
for fine_label, cell_class in braun_fine_to_class.items():
    braun_class_to_region.setdefault(cell_class, []).append(fine_label)

braun_region_star_marg = collapse_fine_props_to_parent_max(
    braun_region_tables["df_marginal_props"], fine_to_parent=braun_fine_to_class, parent_order=braun_tables["df_marginal_props"].columns
)
braun_region_star_cond = collapse_fine_props_to_parent_max(
    braun_region_tables["df_marginal_x_cond_props"], fine_to_parent=braun_fine_to_class, parent_order=braun_tables["df_marginal_x_cond_props"].columns
)

plot_scdrsfm_heatmap(
    df_marginal_props=braun_tables["df_marginal_props"],
    df_intersect_props=braun_tables["df_marginal_x_cond_props"],
    adata=braun_adata,
    biocol="CellClass",
    trait_order=SUBSET_TRAITS,
    trait_labels=TRAIT_LABELS,
    threshold=HEATMAP_THRESHOLD,
    out_png=OUTPUT_DIR / "braun_scdrsfm_cellclass_heatmap_region_stars.png",
    out_csv=BRAUN_SCDRSFM_REGION_STAR_CSV,
    title="",
    df_signal_details=braun_tables["df_signal_details"],
    celltype_order=braun_cellclass_order,
    celltype_groups=braun_cellclass_groups,
    celltype_group_colors=braun_cellclass_colors,
    star_marginal_props=braun_region_star_marg,
    star_conditional_props=braun_region_star_cond,
    star_threshold=REGION_SUPERTYPE_STAR_THRESHOLD,
    annotation_biocol="Region_CellClass",
    annotation_threshold=REGION_SUPERTYPE_STAR_THRESHOLD,
    display_to_annotation_labels=braun_class_to_region,
    fontsize_mult=1.20,
    cbar_label="CellClass prop. sig. conditional cells",
)

Loaded /mnt/shared-workspace/scdrsfm/data/subsets_10k/Braun/human_dev_layers_100k.h5ad: 10,000 cells × 33,538 genes


After filtering: 10,000 cells × 17,536 genes


PASS_Alzheimers_Jansen2019: 490 marginal cells; 118 marginal∩conditional cells; indep assignments raw=118, saved in sections >1.0%=111
PASS_ADHD_Demontis2018: 0 marginal cells; 0 marginal∩conditional cells; indep assignments raw=0, saved in sections >1.0%=0


PASS_BIP_Mullins2021: 3,236 marginal cells; 461 marginal∩conditional cells; indep assignments raw=461, saved in sections >1.0%=461


PASS_Intelligence_SavageJansen2018: 3,454 marginal cells; 160 marginal∩conditional cells; indep assignments raw=160, saved in sections >1.0%=150


...

PASS_MDD_Howard2019: 3,039 marginal cells; 183 marginal∩conditional cells; indep assignments raw=183, saved in sections >1.0%=183


UKB_460K.mental_NEUROTICISM: 2,716 marginal cells; 0 marginal∩conditional cells; indep assignments raw=0, saved in sections >1.0%=0


Saved heatmap cell-type proportions: nature_genetics_manuscript_supplementary/Braun_scDRSFM_cellclass_heatmap_cell_type_proportions.csv (77 rows)
Saved: sea_ad_analysis_outputs/braun_scdrsfm_cellclass_heatmap.png


Saved heatmap cell-type proportions: nature_genetics_manuscript_supplementary/Braun_scDRSFM_cellclass_heatmap_region_stars_cell_type_proportions.csv (84 rows)
Saved: sea_ad_analysis_outputs/braun_scdrsfm_cellclass_heatmap_region_stars.png


["b'Neuron'",
 "b'Radial glia'",
 "b'Neuroblast'",
 "b'Glioblast'",
 "b'Neuronal IPC'",
 "b'Fibroblast'",
 "b'Vascular'",
 "b'Immune'",
 "b'Oligo'",
 "b'Erythrocyte'",
 "b'Placodes'",
 "b'Neural crest'"]

## SEA-AD UMAPs

In [18]:
adata_umap = prepare_umap(adata, force=False)
plot_categorical_umap_with_labels(
    adata_umap,
    color_key="Subclass",
    title="Brain cell types",
    out_png=OUTPUT_DIR / "sea_ad_umap_subclass.png",
    size=6,
    label_fontsize=8,
)


Saved sea_ad_analysis_outputs/sea_ad_umap_subclass.png


## scDRS-FM disease-score and independent-population UMAPs for all traits

In [19]:
scdrsfm_trait_umap_results = {}

for trait in SCDRSFM_TRAITS:
    trait_name = Path(trait).name
    trait_label = TRAIT_LABELS.get(trait, trait_name)

    df_cond_trait, df_marg_trait, signal_remap = assign_conditional_scores_to_cells_filtered_signals(
        adata_umap=adata_umap,
        results_dir=SEA_AD_SCDRSFM_DIR,
        trait=trait,
        indep_sig_col="independent_signal_multi",
        fdr_alpha=FDR_ALPHA,
        min_causal_cells_per_signal=50,
        cellpop_key="Region_Subclass",
        min_fraction_within_any_cellpop=0.01,
    )

    score_key = f"{trait_name}_conditional_score"
    signal_key = f"{trait_name}_independent_signal_multi_filtered"
    out_name = (
        "sea_ad_scdrsfm_ad_conditional_score_and_independent_populations.png"
        if trait == AD_TRAIT
        else f"sea_ad_scdrsfm_{trait_name}_conditional_score_and_independent_populations.png"
    )

    plot_scdrsfm_ad_score_and_signals(
        adata_umap,
        score_key=score_key,
        signal_key=signal_key,
        cellpop_key="Region_Subclass",
        out_png=OUTPUT_DIR / out_name,
        score_title=f"scDRS-FM disease scores\n{trait_label}",
        signal_title=f"scDRS-FM independent populations\n{trait_label}",
        composition_min_ratio=0.01,
        composition_min_cells=50,
        other_alpha=0.3,
        signal_size=14,
        show=(trait == AD_TRAIT),
    )

    scdrsfm_trait_umap_results[trait] = {
        "conditional": df_cond_trait,
        "marginal": df_marg_trait,
        "signal_remap": signal_remap,
        "score_key": score_key,
        "signal_key": signal_key,
        "out_png": OUTPUT_DIR / out_name,
    }

if AD_TRAIT not in scdrsfm_trait_umap_results:
    raise ValueError(f"AD trait {AD_TRAIT!r} is not present in SCDRSFM_TRAITS")

# Preserve the original AD variable names used by the final enrichment workflow.
ad_result = scdrsfm_trait_umap_results[AD_TRAIT]
df_cond_ad = ad_result["conditional"]
df_marg_ad = ad_result["marginal"]
ad_signal_remap = ad_result["signal_remap"]
ad_score_key = ad_result["score_key"]
ad_sig_key = ad_result["signal_key"]
sig_key = ad_sig_key


Saved sea_ad_analysis_outputs/sea_ad_scdrsfm_PASS_Parkinsons23andMe_Corces2020_conditional_score_and_independent_populations.png


Saved sea_ad_analysis_outputs/sea_ad_scdrsfm_ad_conditional_score_and_independent_populations.png


Saved sea_ad_analysis_outputs/sea_ad_scdrsfm_PASS_ADHD_Demontis2018_conditional_score_and_independent_populations.png


Saved sea_ad_analysis_outputs/sea_ad_scdrsfm_PASS_BIP_Mullins2021_conditional_score_and_independent_populations.png


Saved sea_ad_analysis_outputs/sea_ad_scdrsfm_PASS_Intelligence_SavageJansen2018_conditional_score_and_independent_populations.png


Saved sea_ad_analysis_outputs/sea_ad_scdrsfm_PASS_Schizophrenia_Pardinas2018_conditional_score_and_independent_populations.png


Saved sea_ad_analysis_outputs/sea_ad_scdrsfm_PASS_MDD_Howard2019_conditional_score_and_independent_populations.png


Saved sea_ad_analysis_outputs/sea_ad_scdrsfm_UKB_460K.mental_NEUROTICISM_conditional_score_and_independent_populations.png


## Disease-score UMAP comparisons for all traits: scDRS-FM, scDRS, and scPagwas

In [20]:
# ============================================================
# Disease-score UMAP comparisons for every configured trait
# ============================================================
# scDRS-FM marginal:
#   - score from a scDRS-FM marginal score file
#   - significant cells by BH-FDR on marginal p-values
#
# scDRS:
#   - score from a scDRS score file
#   - significant cells by BH-FDR on scDRS p-values
#
# scPagwas:
#   - score from scPagwas.TRS.Score
#   - significant cells by Random_Correct_BG_adjp < 0.1
#   - cell IDs aligned by canonical SEA-AD IDs
#
# Every method is saved as an individual figure, and the original three-panel
# comparison is also saved. Cells are drawn in descending disease-score order.

trait_score_umap_results = {}
disease_umap_files: list[Path] = []

for trait in SUBSET_TRAITS:
    trait_name = Path(trait).name
    trait_label = TRAIT_LABELS.get(trait, trait_name)
    trait_slug = "ad" if trait == AD_TRAIT else trait_name

    scdrsfm_score_file = choose_scdrs_score_file(
        SEA_AD_SCDRSFM_DIR,
        trait,
        suffix_candidates=(".marginal_score.gz", ".score.gz"),
    )
    scdrs_score_file = choose_scdrs_score_file(
        SEA_AD_SCDRS_DIR,
        trait,
        suffix_candidates=(".marginal_score.gz", ".score.gz"),
    )
    scpagwas_score_file = scpagwas_singlecell_file(
        SEA_AD_SCPAGWAS_ROOT,
        trait,
    )

    scdrsfm_marg_scores, scdrsfm_marg_sig = read_score_and_sig_from_scdrs_like_file(
        scdrsfm_score_file,
        adata_umap,
        score_col_candidates=("norm_score", "score", "zscore"),
        pval_col_candidates=("pval", "mc_pval"),
        fdr_alpha=FDR_ALPHA,
    )

    scdrs_scores, scdrs_sig = read_score_and_sig_from_scdrs_like_file(
        scdrs_score_file,
        adata_umap,
        score_col_candidates=("norm_score", "score", "scdrs_score", "zscore"),
        pval_col_candidates=("pval", "mc_pval"),
        fdr_alpha=0.1,
    )

    scpagwas_scores, scpagwas_sig = read_score_and_sig_from_scpagwas_file(
        scpagwas_score_file,
        adata_umap,
        score_col="scPagwas.TRS.Score",
        sig_col="Random_Correct_BG_adjp",
        sig_alpha=0.1,
    )

    method_data = [
        ("scdrsfm", "scDRS-FM marginal disease scores", scdrsfm_marg_scores, scdrsfm_marg_sig),
        ("scdrs", "scDRS disease scores", scdrs_scores, scdrs_sig),
        ("scpagwas", "scPagwas disease scores", scpagwas_scores, scpagwas_sig),
    ]

    method_files: dict[str, Path] = {}
    for method_slug, method_title, method_scores, method_sig in method_data:
        method_out = OUTPUT_DIR / f"sea_ad_{trait_slug}_{method_slug}_disease_scores_highlighted_umap.png"
        plot_score_umap_comparison(
            adata_umap,
            [(f"{method_title}\n{trait_label}", method_scores, method_sig)],
            out_png=method_out,
            show=False,
        )
        method_files[method_slug] = method_out
        disease_umap_files.append(method_out)

    combined_out = (
        OUTPUT_DIR / "sea_ad_ad_score_umap_comparison.png"
        if trait == AD_TRAIT
        else OUTPUT_DIR / f"sea_ad_{trait_name}_score_umap_comparison.png"
    )

    plot_score_umap_comparison(
        adata_umap,
        [
            (f"scDRS-FM marginal scores\n{trait_label}", scdrsfm_marg_scores, scdrsfm_marg_sig),
            (f"scDRS disease scores\n{trait_label}", scdrs_scores, scdrs_sig),
            (f"scPagwas disease scores\n{trait_label}", scpagwas_scores, scpagwas_sig),
        ],
        out_png=combined_out,
        show=(trait == AD_TRAIT),
    )
    disease_umap_files.append(combined_out)

    trait_score_umap_results[trait] = {
        "scdrsfm": (scdrsfm_marg_scores, scdrsfm_marg_sig),
        "scdrs": (scdrs_scores, scdrs_sig),
        "scpagwas": (scpagwas_scores, scpagwas_sig),
        "method_files": method_files,
        "combined_out_png": combined_out,
        # Compatibility with the original notebook key.
        "out_png": combined_out,
    }

print(f"Saved {len(disease_umap_files)} marginal disease-score UMAP figures.")

# Preserve the original AD score variables for interactive inspection.
scdrsfm_marg_scores, scdrsfm_marg_sig = trait_score_umap_results[AD_TRAIT]["scdrsfm"]
scdrs_scores, scdrs_sig = trait_score_umap_results[AD_TRAIT]["scdrs"]
scp_scores, scp_sig = trait_score_umap_results[AD_TRAIT]["scpagwas"]


PASS_Alzheimers_Jansen2019.marginal_score.gz: aligned 10,000 / 10,000 cells
PASS_Alzheimers_Jansen2019.marginal_score.gz: aligned 10,000 / 10,000 cells


Saved sea_ad_analysis_outputs/sea_ad_ad_scdrsfm_disease_scores_highlighted_umap.png


Saved sea_ad_analysis_outputs/sea_ad_ad_scdrs_disease_scores_highlighted_umap.png


Saved sea_ad_analysis_outputs/sea_ad_ad_scpagwas_disease_scores_highlighted_umap.png


Saved sea_ad_analysis_outputs/sea_ad_ad_score_umap_comparison.png
PASS_ADHD_Demontis2018.marginal_score.gz: aligned 10,000 / 10,000 cells
PASS_ADHD_Demontis2018.marginal_score.gz: aligned 10,000 / 10,000 cells


...

Saved sea_ad_analysis_outputs/sea_ad_PASS_MDD_Howard2019_score_umap_comparison.png
UKB_460K.mental_NEUROTICISM.marginal_score.gz: aligned 10,000 / 10,000 cells
UKB_460K.mental_NEUROTICISM.marginal_score.gz: aligned 10,000 / 10,000 cells


Saved sea_ad_analysis_outputs/sea_ad_UKB_460K.mental_NEUROTICISM_scdrsfm_disease_scores_highlighted_umap.png


Saved sea_ad_analysis_outputs/sea_ad_UKB_460K.mental_NEUROTICISM_scdrs_disease_scores_highlighted_umap.png


Saved sea_ad_analysis_outputs/sea_ad_UKB_460K.mental_NEUROTICISM_scpagwas_disease_scores_highlighted_umap.png


Saved sea_ad_analysis_outputs/sea_ad_UKB_460K.mental_NEUROTICISM_score_umap_comparison.png
Saved 28 marginal disease-score UMAP figures.


## Functional phenotype score UMAPs

In [21]:
phenotype_umap_scores = {}

for phenotype in FUNCTIONAL_PHENOTYPES:
    score_file = choose_scdrs_score_file(
        SEA_AD_PHENOTYPE_SCORE_DIR,
        phenotype,
        suffix_candidates=(".marginal_score.gz", ".score.gz"),
    )
    score_multiplier = PHENOTYPE_SCORE_MULTIPLIERS.get(phenotype, 1.0)
    phenotype_label = PHENOTYPE_LABELS.get(phenotype, phenotype)
    phenotype_token = phenotype.removesuffix("_gs").lower()
    out_name = (
        "sea_ad_hm_negative_phenotype_umap.png"
        if phenotype == "HM_gs"
        else f"sea_ad_{phenotype_token}_phenotype_umap.png"
    )

    phenotype_umap_scores[phenotype] = plot_score_file_umap(
        adata_umap,
        score_file,
        score_col="norm_score",
        multiply_by=score_multiplier,
        title=f"{phenotype_label} phenotype scores",
        out_png=OUTPUT_DIR / out_name,
        cell_type="Microglia-PVM",
        show=(phenotype == "HM_gs"),
    )

hm_scores = phenotype_umap_scores["HM_gs"]


Saved sea_ad_analysis_outputs/sea_ad_hm_negative_phenotype_umap.png


Saved sea_ad_analysis_outputs/sea_ad_dam_phenotype_umap.png


Saved sea_ad_analysis_outputs/sea_ad_crm_phenotype_umap.png


Saved sea_ad_analysis_outputs/sea_ad_irm_phenotype_umap.png


Saved sea_ad_analysis_outputs/sea_ad_hla_phenotype_umap.png


## Functional phenotype gene sets and per-signal enrichment

In [22]:
functional_phenotypes = list(FUNCTIONAL_PHENOTYPES)
MICROGLIA_PHENOTYPE_DIR = Path("microglia")


def read_geneset_file(path: Path) -> tuple[str, list[str]]:
    df = pd.read_csv(path, sep="	")
    if not {"TRAIT", "GENESET"}.issubset(df.columns):
        raise ValueError(f"{path} missing required columns TRAIT and GENESET")
    trait = str(df.loc[0, "TRAIT"])
    genes = [g.strip() for g in str(df.loc[0, "GENESET"]).split(",") if str(g).strip()]
    return trait, genes


def load_functional_genesets(phenotypes: Sequence[str], folder: Path):
    rows_long = []
    rows_wide = []
    genesets = {}
    for phenotype in phenotypes:
        candidates = sorted(Path(folder).glob(f"{phenotype}*"))
        if not candidates:
            raise FileNotFoundError(f"No file found for phenotype {phenotype!r} in {folder}")
        path = candidates[0]
        trait, genes = read_geneset_file(path)
        genesets[phenotype] = genes
        rows_wide.append({
            "phenotype": phenotype,
            "trait_in_file": trait,
            "file": path.name,
            "n_genes_listed": len(genes),
            "geneset": ",".join(genes),
        })
        for gene in genes:
            rows_long.append({"phenotype": phenotype, "file": path.name, "gene": gene})
    return genesets, pd.DataFrame(rows_long), pd.DataFrame(rows_wide)


def add_mean_geneset_scores(adata: sc.AnnData, genesets: Mapping[str, Sequence[str]], *, score_prefix="microglia_pheno_", zscore=True):
    X = adata.X
    var_upper = pd.Index(adata.var_names).str.upper()
    gene_to_idx = {gene: i for i, gene in enumerate(var_upper)}
    missing = {}
    for phenotype, genes in genesets.items():
        idx = []
        missing[phenotype] = []
        for gene in genes:
            gene_upper = str(gene).upper()
            if gene_upper in gene_to_idx:
                idx.append(gene_to_idx[gene_upper])
            else:
                missing[phenotype].append(gene)
        col = f"{score_prefix}{phenotype}"
        if not idx:
            adata.obs[col] = np.nan
            continue
        if sp.issparse(X):
            scores = np.asarray(X[:, idx].mean(axis=1)).ravel()
        else:
            scores = np.mean(X[:, idx], axis=1)
        if zscore:
            mu = np.nanmean(scores)
            sd = np.nanstd(scores)
            scores = (scores - mu) / (sd if sd > 0 else 1.0)
        adata.obs[col] = scores
    return missing


def rank_biserial_from_mwu(U, n1, n2):
    auc = U / (n1 * n2)
    return 2 * auc - 1


def bh_fdr_values(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan, dtype=float)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out
    _, q, _, _ = multipletests(p[ok], alpha=0.05, method="fdr_bh")
    out[np.where(ok)[0]] = q
    return out


genesets, geneset_long_df, geneset_wide_df = load_functional_genesets(functional_phenotypes, MICROGLIA_PHENOTYPE_DIR)
missing_by_phenotype = add_mean_geneset_scores(adata_umap, genesets, score_prefix="microglia_pheno_", zscore=True)

signal_values = pd.to_numeric(adata_umap.obs[sig_key].astype(str), errors="coerce").fillna(-1).astype(int)
signals = sorted([s for s in signal_values.unique() if s >= 1])

results = []
for signal in signals:
    sig_mask = (signal_values == signal).to_numpy()
    bg_mask = ~sig_mask
    pvals = []
    tmp_rows = []
    for phenotype in functional_phenotypes:
        col = f"microglia_pheno_{phenotype}"
        x1 = adata_umap.obs.loc[sig_mask, col].to_numpy(dtype=float)
        x2 = adata_umap.obs.loc[bg_mask, col].to_numpy(dtype=float)
        x1 = x1[np.isfinite(x1)]
        x2 = x2[np.isfinite(x2)]
        if len(x1) == 0 or len(x2) == 0:
            U = p = rbc = dmean = np.nan
        else:
            U, p = mannwhitneyu(x1, x2, alternative="two-sided")
            rbc = rank_biserial_from_mwu(U, len(x1), len(x2))
            dmean = float(np.mean(x1) - np.mean(x2))
        pvals.append(p)
        tmp_rows.append((phenotype, rbc, dmean, p, len(x1), len(x2)))
    qvals = bh_fdr_values(np.array(pvals, dtype=float))
    for idx, (phenotype, rbc, dmean, p, n_sig, n_bg) in enumerate(tmp_rows):
        results.append({
            "signal": signal,
            "phenotype": phenotype,
            "n_signal": n_sig,
            "n_background": n_bg,
            "rank_biserial": rbc,
            "delta_mean": dmean,
            "pval": p,
            "qval": qvals[idx],
        })

enrich_df = pd.DataFrame(results)
display(enrich_df.head())

,signal,phenotype,n_signal,n_background,rank_biserial,delta_mean,pval,qval
0,1,HM_gs,228,9772,0.910295,2.750809,1.867925e-122,2.334906e-122
1,1,DAM_gs,228,9772,0.942506,3.091789,3.936252e-131,6.560419e-131
2,1,CRM_gs,228,9772,0.964232,3.534038,3.742611e-137,1.871306e-136
3,1,IRM_gs,228,9772,0.954914,3.385011,1.487843e-134,3.719608e-134
4,1,HLA_gs,228,9772,0.891459,2.293437,1.606401e-117,1.606401e-117


## Functional phenotype enrichment heatmap

In [23]:
effect_mat = enrich_df.pivot(index="phenotype", columns="signal", values="rank_biserial").reindex(index=functional_phenotypes, columns=signals)
q_mat = enrich_df.pivot(index="phenotype", columns="signal", values="qval").reindex(index=functional_phenotypes, columns=signals)

fig, ax = plt.subplots(figsize=(max(7, len(signals) * 0.9 + 4), max(5, len(functional_phenotypes) * 0.45 + 2.5)))
sns.heatmap(
    effect_mat,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.3,
    square=True,
    cbar_kws={"label": "Enrichment (rank-biserial; signal vs all other cells)"},
    ax=ax,
)
for i, phenotype in enumerate(effect_mat.index):
    for j, signal in enumerate(effect_mat.columns):
        q = q_mat.loc[phenotype, signal]
        if pd.notna(q) and q < 0.05:
            ax.text(j + 0.5, i + 0.5, "★", ha="center", va="center", fontsize=11)
ax.set_xlabel("Independent signal")
ax.set_ylabel("Functional phenotype")
ax.set_title("Functional phenotype enrichment per independent signal")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sea_ad_functional_phenotype_enrichment_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## Final pathway heatmap setup

This section keeps the final pathway heatmap workflow from the original notebook, but uses cleaned helper functions and a robust Enrichr loader.

In [24]:
# Compute the five DEG comparisons used by the final pathway heatmap

import numpy as np
import pandas as pd
import scanpy as sc

# ============================================================
# Parameters
# ============================================================
signal_col = sig_key              # replace if needed
subclass_col = "Subclass"
region_subclass_col = "Region_Subclass"

alpha = 0.05
lfc_min = 0.25
use_raw = False
layer = None

# ============================================================
# Region_Subclass target definitions
# ============================================================
microglia_region_labels = [
    "MEC_Microglia-PVM",
    "MTG_Microglia-PVM",
    "DFC_Microglia-PVM",
]

human_mtg_it_region_labels = [
    "Human MTG_L6 IT",
    "Human MTG_L2/3 IT",
]

# ============================================================
# Helper
# ============================================================
def run_target_vs_rest_in_subset(
    adata,
    target_mask,
    subset_mask,
    comparison_name,
    target_name,
    subset_name,
    alpha=0.05,
    lfc_min=0.25,
    use_raw=False,
    layer=None,
):
    """
    Run DE for:
        target cells
        vs
        all other cells
    within the subset defined by subset_mask.
    """
    target_mask = pd.Series(target_mask, index=adata.obs_names).fillna(False).astype(bool)
    subset_mask = pd.Series(subset_mask, index=adata.obs_names).fillna(False).astype(bool)

    if subset_mask.sum() == 0:
        raise ValueError(f"{comparison_name}: no cells found in subset '{subset_name}'")

    ad = adata[subset_mask].copy()
    target_in_subset = target_mask.loc[ad.obs_names].astype(bool)

    n_target = int(target_in_subset.sum())
    n_rest = int((~target_in_subset).sum())

    if n_target == 0:
        raise ValueError(f"{comparison_name}: no target cells found inside subset '{subset_name}'")
    if n_rest == 0:
        raise ValueError(f"{comparison_name}: no rest cells found inside subset '{subset_name}'")

    group_key = f"deg_group_{comparison_name}"
    ad.obs[group_key] = pd.Categorical(
        np.where(target_in_subset, "Target", "Rest"),
        categories=["Target", "Rest"],
        ordered=True,
    )

    sc.tl.rank_genes_groups(
        ad,
        groupby=group_key,
        groups=["Target"],
        reference="Rest",
        method="wilcoxon",
        use_raw=use_raw,
        layer=layer,
        n_genes=ad.n_vars,
    )

    df = sc.get.rank_genes_groups_df(ad, group="Target")

    keep = df["pvals_adj"].notna() & (df["pvals_adj"] < alpha)
    if "logfoldchanges" in df.columns and df["logfoldchanges"].notna().any():
        keep &= df["logfoldchanges"].abs() >= lfc_min

    df_sig = df.loc[keep].copy()
    df_sig["comparison"] = comparison_name
    df_sig["target_name"] = target_name
    df_sig["subset_name"] = subset_name
    df_sig["n_target_cells"] = n_target
    df_sig["n_rest_cells"] = n_rest

    return {
        "adata_subset": ad,
        "all_ranked_genes": df,
        "sig_degs": df_sig,
        "deg_set": set(df_sig["names"].astype(str)),
        "n_target_cells": n_target,
        "n_rest_cells": n_rest,
        "target_name": target_name,
        "subset_name": subset_name,
    }

# ============================================================
# Precompute masks
# ============================================================
subclass_vals = adata_umap.obs[subclass_col].astype(str)
region_subclass_vals = adata_umap.obs[region_subclass_col].astype(str)

sig_raw = (
    pd.to_numeric(adata_umap.obs[signal_col].astype(str), errors="coerce")
    .fillna(-1)
    .astype(int)
)

# Universes
microglia_subset_mask = subclass_vals == "Microglia-PVM"

it_subset_mask = (
    subclass_vals.str.contains("L6 IT", regex=False, na=False)
    | subclass_vals.str.contains("L2/3 IT", regex=False, na=False)
)

# Targets
signal1_mask = sig_raw == 1
signal2_mask = sig_raw == 2
signal3_mask = sig_raw == 3

# Original SEA-AD procedure: these regional comparison targets include all
# cells in the named region/subclass groups. Do not exclude retained signal
# cells with an additional ``sig_raw == -1`` filter.
microglia_regions_union_mask = region_subclass_vals.isin(microglia_region_labels)
human_mtg_it_regions_union_mask = region_subclass_vals.isin(human_mtg_it_region_labels)

# ============================================================
# Run the 5 DEG analyses
# ============================================================
deg_results = {}

comparison_specs = [
    {
        "key": "signal1_vs_rest_microglia_union",
        "target_mask": signal1_mask,
        "subset_mask": microglia_subset_mask,
        "target_name": "signal 1",
        "subset_name": 'Subclass == "Microglia-PVM"',
    },
    {
        "key": "signal2_vs_rest_microglia_union",
        "target_mask": signal2_mask,
        "subset_mask": microglia_subset_mask,
        "target_name": "signal 2",
        "subset_name": 'Subclass == "Microglia-PVM"',
    },
    {
        "key": "microglia_regions_vs_rest_microglia_union",
        "target_mask": microglia_regions_union_mask,
        "subset_mask": microglia_subset_mask,
        "target_name": "MEC/MTG/DFC Microglia-PVM union",
        "subset_name": 'Subclass == "Microglia-PVM"',
    },
    {
        "key": "signal3_vs_rest_IT_union",
        "target_mask": signal3_mask,
        "subset_mask": it_subset_mask,
        "target_name": "signal 3",
        "subset_name": 'Subclass contains "L6 IT" or "L2/3 IT"',
    },
    {
        "key": "human_mtg_it_regions_vs_rest_IT_union",
        "target_mask": human_mtg_it_regions_union_mask,
        "subset_mask": it_subset_mask,
        "target_name": "Human MTG L6 IT + L2/3 IT union",
        "subset_name": 'Subclass contains "L6 IT" or "L2/3 IT"',
    },
]

# ---------------------------------------------------------------------------
# [10k-scale note] scDRS-FM package code is UNCHANGED. This is a notebook-only
# guard for the 10k-cell subset: some independent-signal populations that exist
# at full scale (e.g. signal 2 / signal 3) may contain zero cells here, which
# would make run_target_vs_rest_in_subset raise "no target cells found".
# We skip any comparison whose target has 0 cells inside its subset (instead of
# crashing) and report it, so the downstream pathway heatmap is built from the
# comparisons that are actually populated at 10k scale.
# ---------------------------------------------------------------------------
skipped_comparisons = []
for spec in comparison_specs:
    print(f"Running {spec['key']}")
    _tmask = pd.Series(spec["target_mask"], index=adata_umap.obs_names).fillna(False).astype(bool)
    _smask = pd.Series(spec["subset_mask"], index=adata_umap.obs_names).fillna(False).astype(bool)
    _n_target_in_subset = int((_tmask & _smask).sum())
    if _smask.sum() == 0 or _n_target_in_subset == 0:
        print(f"  [10k-scale note] skipping {spec['key']}: "
              f"{_n_target_in_subset} target cells inside subset "
              f"(subset size={int(_smask.sum())}) at 10k scale")
        skipped_comparisons.append(spec["key"])
        continue
    deg_results[spec["key"]] = run_target_vs_rest_in_subset(
        adata=adata_umap,
        target_mask=spec["target_mask"],
        subset_mask=spec["subset_mask"],
        comparison_name=spec["key"],
        target_name=spec["target_name"],
        subset_name=spec["subset_name"],
        alpha=alpha,
        lfc_min=lfc_min,
        use_raw=use_raw,
        layer=layer,
    )
if skipped_comparisons:
    print(f"[10k-scale note] skipped {len(skipped_comparisons)} empty comparison(s): {skipped_comparisons}")

# ============================================================
# Collect outputs
# ============================================================
deg_tables = []
deg_sets = {}

for name, res in deg_results.items():
    tmp = res["sig_degs"].copy()
    deg_tables.append(tmp)
    deg_sets[name] = res["deg_set"]

deg_df = pd.concat(deg_tables, ignore_index=True) if deg_tables else pd.DataFrame()

# ============================================================
# Summary
# ============================================================
for name, res in deg_results.items():
    print(
        f"{name}: "
        f"{res['n_target_cells']} target cells vs "
        f"{res['n_rest_cells']} rest cells | "
        f"{len(res['sig_degs'])} significant DEGs"
    )
    print(f"  Target: {res['target_name']}")
    print(f"  Universe: {res['subset_name']}")

display(deg_df.head())

print("\ndeg_results.keys():")
print(deg_results.keys())

Running signal1_vs_rest_microglia_union


Running signal2_vs_rest_microglia_union
  [10k-scale note] skipping signal2_vs_rest_microglia_union: 0 target cells inside subset (subset size=1332) at 10k scale
Running microglia_regions_vs_rest_microglia_union


Running signal3_vs_rest_IT_union
  [10k-scale note] skipping signal3_vs_rest_IT_union: 0 target cells inside subset (subset size=2543) at 10k scale
Running human_mtg_it_regions_vs_rest_IT_union


[10k-scale note] skipped 2 empty comparison(s): ['signal2_vs_rest_microglia_union', 'signal3_vs_rest_IT_union']
signal1_vs_rest_microglia_union: 228 target cells vs 1104 rest cells | 2766 significant DEGs
  Target: signal 1
  Universe: Subclass == "Microglia-PVM"
microglia_regions_vs_rest_microglia_union: 792 target cells vs 540 rest cells | 1832 significant DEGs
  Target: MEC/MTG/DFC Microglia-PVM union
  Universe: Subclass == "Microglia-PVM"
human_mtg_it_regions_vs_rest_IT_union: 1171 target cells vs 1372 rest cells | 2706 significant DEGs
  Target: Human MTG L6 IT + L2/3 IT union
  Universe: Subclass contains "L6 IT" or "L2/3 IT"


,names,scores,logfoldchanges,pvals,pvals_adj,comparison,target_name,subset_name,n_target_cells,n_rest_cells
0,TBC1D14,12.888683,2.504651,5.212591e-38,1.234602e-33,signal1_vs_rest_microglia_union,signal 1,"Subclass == ""Microglia-PVM""",228,1104
1,DPYD,12.257610,2.880717,1.529437e-34,1.811236e-30,signal1_vs_rest_microglia_union,signal 1,"Subclass == ""Microglia-PVM""",228,1104
2,SLC11A1,12.112087,2.930855,9.114490e-34,7.195889e-30,signal1_vs_rest_microglia_union,signal 1,"Subclass == ""Microglia-PVM""",228,1104
3,ARHGAP26,12.033227,1.364972,2.376864e-33,1.407400e-29,signal1_vs_rest_microglia_union,signal 1,"Subclass == ""Microglia-PVM""",228,1104
4,KCNMA1,11.828133,1.898650,2.792860e-32,1.322978e-28,signal1_vs_rest_microglia_union,signal 1,"Subclass == ""Microglia-PVM""",228,1104



deg_results.keys():
dict_keys(['signal1_vs_rest_microglia_union', 'microglia_regions_vs_rest_microglia_union', 'human_mtg_it_regions_vs_rest_IT_union'])


## Robust Enrichr library loader

In [25]:
import requests

GENESET_CACHE_DIR = Path("geneset_cache")
GENESET_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def canonicalize_geneset_dict(gs_dict):
    out = {}
    for term, genes in gs_dict.items():
        clean = sorted({str(g).strip().upper() for g in genes if pd.notna(g) and str(g).strip()})
        if clean:
            out[str(term)] = clean
    return out


def safe_cache_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name)


def fetch_enrichr_library_direct(lib_name: str, organism: str = "Human") -> dict[str, list[str]]:
    base_url = "https://maayanlab.cloud/Enrichr" if organism.lower() == "human" else "https://maayanlab.cloud/MouseEnrichr"
    response = requests.get(f"{base_url}/geneSetLibrary", params={"mode": "text", "libraryName": lib_name}, timeout=120)
    response.raise_for_status()
    text = response.content.decode("utf-8", errors="replace")
    gene_sets = {}
    for line in text.splitlines():
        parts = line.strip().split("	")
        if len(parts) < 3:
            continue
        term = parts[0]
        genes = [field.split(",")[0].strip() for field in parts[2:] if field.strip()]
        if genes:
            gene_sets[term] = genes
    return canonicalize_geneset_dict(gene_sets)


def load_or_fetch_enrichr_library(lib_name: str, organism: str = "Human", force_refresh: bool = False):
    cache_file = GENESET_CACHE_DIR / f"{safe_cache_name(organism + '_' + lib_name)}.pkl"
    if cache_file.exists() and not force_refresh:
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    gene_sets = fetch_enrichr_library_direct(lib_name, organism=organism)
    with open(cache_file, "wb") as f:
        pickle.dump(gene_sets, f)
    return gene_sets


def _upper_set(xs):
    return set(pd.Index(list(xs)).astype(str).str.upper())


def parse_geneset_string(x):
    if pd.isna(x):
        return set()
    return {g.strip().upper() for g in str(x).split(",") if str(g).strip()}


PHENOTYPE_GENESETS = (
    geneset_wide_df.groupby("phenotype")["geneset"]
    .apply(lambda s: set().union(*[parse_geneset_string(x) for x in s]))
    .to_dict()
)
GENE_UNIVERSE = _upper_set(adata.var_names)

GO_BP_NAME = "GO_Biological_Process_2026"
GO_CC_NAME = "GO_Cellular_Component_2026"
GO_MF_NAME = "GO_Molecular_Function_2026"
KEGG_NAME = "KEGG_2026"
REACTOME_NAME = "Reactome_Pathways_2024"

print("Using libraries:")
print("  GO BP     :", GO_BP_NAME)
print("  GO CC     :", GO_CC_NAME)
print("  GO MF     :", GO_MF_NAME)
print("  KEGG      :", KEGG_NAME)
print("  Reactome  :", REACTOME_NAME)

GO_BP = load_or_fetch_enrichr_library(GO_BP_NAME)
GO_CC = load_or_fetch_enrichr_library(GO_CC_NAME)
GO_MF = load_or_fetch_enrichr_library(GO_MF_NAME)
KEGG_PATHWAYS = load_or_fetch_enrichr_library(KEGG_NAME)
REACTOME_PATHWAYS = load_or_fetch_enrichr_library(REACTOME_NAME)

GO_PATHWAYS = {}
GO_PATHWAYS.update({f"BP: {k}": v for k, v in GO_BP.items()})
GO_PATHWAYS.update({f"CC: {k}": v for k, v in GO_CC.items()})
GO_PATHWAYS.update({f"MF: {k}": v for k, v in GO_MF.items()})

print(f"PHENOTYPE_GENESETS: {len(PHENOTYPE_GENESETS):,}")
print(f"GO_PATHWAYS:        {len(GO_PATHWAYS):,}")
print(f"KEGG_PATHWAYS:      {len(KEGG_PATHWAYS):,}")
print(f"REACTOME_PATHWAYS:  {len(REACTOME_PATHWAYS):,}")

Using libraries:
  GO BP     : GO_Biological_Process_2026
  GO CC     : GO_Cellular_Component_2026
  GO MF     : GO_Molecular_Function_2026
  KEGG      : KEGG_2026
  Reactome  : Reactome_Pathways_2024
PHENOTYPE_GENESETS: 5
GO_PATHWAYS:        7,858
KEGG_PATHWAYS:      352
REACTOME_PATHWAYS:  2,105


## Final pathway heatmap

In [26]:
# ORA + 5-row pathway heatmap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
import seaborn as sns
import gseapy as gp
import textwrap

# ============================================================
# Parameters
# ============================================================
MIN_GENESET_SIZE = 50
MAX_GENESET_SIZE = 500
MIN_RECALL = 0.0
MIN_INTERSECT = 0
TOP_N_PER_POPULATION = 3  # [10k-scale note] fewer top pathways per population to de-crowd x-axis (package code unchanged)

DE_PADJ_CUTOFF = 0.05
ENRICH_FDR_CUTOFF = 0.05

SAVEPATH = str(OUTPUT_DIR / "sea_ad_five_rows_five_column_groups_heatmap.png")   # set to None to skip saving
DPI = 300

OR_VMIN = 0.1
OR_VCENTER = 1.0
OR_VMAX = 3.5

YTICK_FONTSIZE = 20
XTICK_FONTSIZE = 11  # [10k-scale note] reduced from 16 to de-crowd pathway x-labels (package code unchanged)
ANNOT_FONTSIZE = 15
CBAR_LABEL_FONTSIZE = 20
CBAR_TICK_FONTSIZE = 18
TOPBAR_LABEL_FONTSIZE = 16

ANNOT_FONTWEIGHT = "semibold"
TOPBAR_LABEL_FONTWEIGHT = "bold"

XTICK_ROTATION = 90  # [10k-scale note] vertical labels to prevent overlap with many pathway columns (package code unchanged)
TERM_WRAP_WIDTH = 16  # [10k-scale note] tighter wrap for readability (package code unchanged)

ANNOT_OUTLINE_COLOR = "black"
ANNOT_OUTLINE_WIDTH = 2.0

mapping_dict = {
    "BP: Heterophilic Cell-Cell Adhesion (GO:0007157)": "Heterophilic adhesion",
    "MF: Transmembrane Receptor Protein Tyrosine Kinase Activity (GO:0004714)": "Receptor tyrosine kinase activity",
    "MF: Growth Factor Activity (GO:0008083)": "Growth factor activity",
    "BP: Neurogenesis (GO:0022008)": "Neurogenesis",
    "BP: Cell-cell Junction Organization (GO:0045216)": "Cell junction organization",
    "REACTOME: FCGR Activation": "FCGR activation",
    "REACTOME: Transcriptional and Post-Translational Regulation of MITF-M Expression and Activity": "MITF-M regulation",
    "BP: Positive Regulation of miRNA Metabolic Process (GO:2000630)": "miRNA metabolism upregulation",
    "CC: Microvillus (GO:0005902)": "Microvillus",
    "BP: Blood Vessel Morphogenesis (GO:0048514)": "Blood vessel morphogenesis",
    "BP: Positive Regulation of Leukocyte Cell-Cell Adhesion (GO:1903039)": "Leukocyte adhesion upregulation",
    "CC: Clathrin-coated Endocytic Vesicle Membrane (GO:0030669)": "Clathrin-coated vesicle membrane",
    "KEGG: SYSTEMIC LUPUS ERYTHEMATOSUS": "Systemic lupus",
    "KEGG: FANCONI ANEMIA PATHWAY": "Fanconi anemia pathway",
    "MF: Calcium Channel Activity (GO:0005262)": "Calcium channel activity",
    "CC: Intermediate Filament Cytoskeleton (GO:0045111)": "Intermediate filament cytoskeleton",
    "REACTOME: Collagen Biosynthesis and Modifying Enzymes": "Collagen biosynthesis",
    "REACTOME: Collagen Formation": "Collagen formation",
    "BP: Response to Retinoic Acid (GO:0032526)": "Retinoic acid response",
    "REACTOME: IRS-related Events Triggered by IGF1R": "IGF1R–IRS signaling",
    "BP: Regulation of Neurogenesis (GO:0050767)": "Neurogenesis regulation",
    "MF: E-box Binding (GO:0070888)": "E-box binding",
    "CC: Motile Cilium (GO:0031514)": "Motile cilium",
    "BP: Organic Anion Transport (GO:0015711)": "Organic anion transport",

}

# ============================================================
# Rows and columns
# ============================================================
COMPARISONS = [
    {
        "key": "signal1_vs_rest_microglia_union",
        "title": "Population 1",
        "topbar_label": "Top pathways for \npopulation 1",
        "topbar_color": "#1f77b4",
    },
    {
        "key": "signal2_vs_rest_microglia_union",
        "title": "Population 2",
        "topbar_label": "Top pathways for \npopulation 2",
        "topbar_color": "#d62728",
    },
    {
        "key": "microglia_regions_vs_rest_microglia_union",
        "title": "All non sig. \n MEC/DFC/MTG microglia",
        "topbar_label": "Top pathways for \nnon sig. MEC/DFC/MTG microglia",
        "topbar_color": "#9467bd",
    },
    {
        "key": "signal3_vs_rest_IT_union",
        "title": "Population 3",
        "topbar_label": "Top pathways for \npopulation 3",
        "topbar_color": "#2ca02c",
    },
    {
        "key": "human_mtg_it_regions_vs_rest_IT_union",
        "title": "All non sig. \nMTG IT neurons",
        "topbar_label": "Top pathways for \nnon sig. MTG IT neurons",
        "topbar_color": "#8c564b",
    },
]

# Include the background cell sets as column groups too
COLUMN_COMPARISONS = [
    {
        "key": "signal1_vs_rest_microglia_union",
        "title": "Population 1",
        "topbar_label": "Top pathways for \npopulation 1",
        "topbar_color": "#1f77b4",
    },
    {
        "key": "signal2_vs_rest_microglia_union",
        "title": "Population 2",
        "topbar_label": "Top pathways for \npopulation 2",
        "topbar_color": "#d62728",
    },
    {
        "key": "microglia_regions_vs_rest_microglia_union",
        "title": "All non sig. \nMEC/DFC/MTG microglia",
        "topbar_label": "Top pathways for \nnon sig. MEC/DFC/\nMTG microglia",
        "topbar_color": "#9467bd",
    },
    {
        "key": "signal3_vs_rest_IT_union",
        "title": "Population 3",
        "topbar_label": "Top pathways for \npopulation 3",
        "topbar_color": "#2ca02c",
    },
    {
        "key": "human_mtg_it_regions_vs_rest_IT_union",
        "title": "All non sig.\nMTG IT neurons",
        "topbar_label": "Top pathways for \nnon sig. MTG IT\n neurons",
        "topbar_color": "#8c564b",
    },
]

# ---------------------------------------------------------------------------
# [10k-scale note] scDRS-FM package code is UNCHANGED. Prune the hardcoded
# comparison lists to the comparisons that were actually computed in the
# previous cell (some independent-signal populations, e.g. signal 2 / signal 3,
# have zero cells at 10k scale and were skipped). This keeps the pathway heatmap
# and ORA loops from KeyError-ing on comparisons that do not exist at 10k scale.
# ---------------------------------------------------------------------------
_present_keys = set(deg_results.keys())
_orig_n = len(COMPARISONS)
COMPARISONS = [c for c in COMPARISONS if c["key"] in _present_keys]
COLUMN_COMPARISONS = [c for c in COLUMN_COMPARISONS if c["key"] in _present_keys]
if len(COMPARISONS) < _orig_n:
    print(f"[10k-scale note] pathway-heatmap comparisons pruned to computed keys: "
          f"{[c['key'] for c in COMPARISONS]}")

# ============================================================
# All genesets
# Assumes these already exist:
# PHENOTYPE_GENESETS, GO_PATHWAYS, KEGG_PATHWAYS, REACTOME_PATHWAYS,
# canonicalize_geneset_dict
# ============================================================
ALL_GENESETS = {}
ALL_GENESETS.update({f"PHENO: {k}": v for k, v in PHENOTYPE_GENESETS.items()})
ALL_GENESETS.update({f"{k}": v for k, v in GO_PATHWAYS.items()})
ALL_GENESETS.update({f"KEGG: {k}": v for k, v in KEGG_PATHWAYS.items()})
ALL_GENESETS.update({f"REACTOME: {k}": v for k, v in REACTOME_PATHWAYS.items()})

ALL_GENESETS = canonicalize_geneset_dict(ALL_GENESETS)
ALL_GENESETS = {
    k: v for k, v in ALL_GENESETS.items()
    if (len(v) > MIN_GENESET_SIZE) and (len(v) < MAX_GENESET_SIZE)
}

# ============================================================
# Helpers
# ============================================================
def prep_ranked_de_table(df):
    out = df.copy()
    out["names"] = out["names"].astype(str).str.upper().str.strip()
    out = out.loc[out["names"].notna() & (out["names"] != "")]
    return out


def split_pos_neg_deg_genes(df, padj_cutoff=0.05):
    d = prep_ranked_de_table(df)
    d = d.loc[d["pvals_adj"].notna() & (d["pvals_adj"] < padj_cutoff)].copy()

    if "logfoldchanges" not in d.columns:
        raise ValueError("Expected 'logfoldchanges' column in DE table.")

    pos = d.loc[d["logfoldchanges"] > 0, "names"].drop_duplicates().tolist()
    neg = d.loc[d["logfoldchanges"] < 0, "names"].drop_duplicates().tolist()
    return pos, neg


def parse_overlap_string(x):
    if pd.isna(x):
        return (np.nan, np.nan)

    if isinstance(x, str) and "/" in x:
        a, b = x.split("/", 1)
        try:
            return (float(a), float(b))
        except ValueError:
            return (np.nan, np.nan)

    return (np.nan, np.nan)


def run_enrichr_dict(
    gene_list,
    genesets_dict,
    background_genes=None,
    enrich_fdr_cutoff=0.05,
    min_recall=0.10,
    min_intersect=10,
):
    empty_cols = [
        "Term", "Odds Ratio", "Adjusted P-value", "P-value",
        "intersect", "geneset_size", "recall", "is_significant"
    ]
    if len(gene_list) == 0:
        return pd.DataFrame(columns=empty_cols)

    enr = gp.enrich(
        gene_list=gene_list,
        gene_sets=genesets_dict,
        background=background_genes,
        outdir=None,
        no_plot=True,
        verbose=False,
    )

    if enr.results is None or len(enr.results) == 0:
        return pd.DataFrame(columns=empty_cols)

    res = enr.results.copy()

    for col in ["Adjusted P-value", "P-value", "Odds Ratio", "Combined Score"]:
        if col in res.columns:
            res[col] = pd.to_numeric(res[col], errors="coerce")

    res = res.loc[res["Term"].notna() & res["Odds Ratio"].notna()].copy()
    res = res.loc[res["Odds Ratio"] > 0].copy()

    res["geneset_size"] = res["Term"].map(lambda x: len(genesets_dict.get(x, [])))

    if "Overlap" in res.columns:
        parsed = res["Overlap"].apply(parse_overlap_string)
        res["intersect"] = parsed.apply(lambda x: x[0])
        overlap_denom = parsed.apply(lambda x: x[1])

        missing_mask = res["geneset_size"].isna()
        res.loc[missing_mask, "geneset_size"] = overlap_denom[missing_mask]
    else:
        res["intersect"] = np.nan

    res["geneset_size"] = pd.to_numeric(res["geneset_size"], errors="coerce")
    res["intersect"] = pd.to_numeric(res["intersect"], errors="coerce")
    res["recall"] = res["intersect"] / res["geneset_size"]

    res = res.loc[
        res["geneset_size"].notna()
        & (res["geneset_size"] > MIN_GENESET_SIZE)
        & (res["geneset_size"] < MAX_GENESET_SIZE)
        & res["intersect"].notna()
        & (res["intersect"] > min_intersect)
        & res["recall"].notna()
        & (res["recall"] > min_recall)
    ].copy()

    res["is_significant"] = res["Adjusted P-value"] < enrich_fdr_cutoff
    return res


def select_top_n_unique_sig_terms(target_key, enrichment_results, comparisons_for_uniqueness, top_n=10):
    target_res = enrichment_results[target_key].copy()
    if target_res.empty:
        return []

    target_sig = target_res.loc[target_res["is_significant"]].copy()
    if target_sig.empty:
        return []

    other_sig_terms = set()
    for comp in comparisons_for_uniqueness:
        other_key = comp["key"]
        if other_key == target_key:
            continue

        other_res = enrichment_results[other_key]
        if other_res.empty:
            continue

        other_sig_terms.update(
            other_res.loc[other_res["is_significant"], "Term"].astype(str).tolist()
        )

    target_unique = target_sig.loc[
        ~target_sig["Term"].astype(str).isin(other_sig_terms)
    ].copy()

    if target_unique.empty:
        return []

    ranked = (
        target_unique.sort_values(
            ["Odds Ratio", "Adjusted P-value", "P-value", "Term"],
            ascending=[False, True, True, True]
        )["Term"]
        .head(top_n)
        .tolist()
    )
    return ranked


def build_grouped_topn_table_multi(enrichment_results, all_comparisons, column_comparisons, top_n=10):
    merged = None

    for comp in all_comparisons:
        key = comp["key"]
        res = enrichment_results[key][[
            "Term", "Odds Ratio", "Adjusted P-value", "is_significant",
            "intersect", "geneset_size", "recall"
        ]].copy().rename(columns={
            "Odds Ratio": f"{key}_or",
            "Adjusted P-value": f"{key}_fdr",
            "is_significant": f"{key}_sig",
            "intersect": f"{key}_intersect",
            "geneset_size": f"{key}_geneset_size",
            "recall": f"{key}_recall",
        })

        merged = res if merged is None else merged.merge(res, on="Term", how="outer")

    if merged is None or merged.empty:
        return pd.DataFrame()

    for comp in all_comparisons:
        key = comp["key"]
        merged[f"{key}_or"] = merged[f"{key}_or"].fillna(1.0)
        merged[f"{key}_fdr"] = merged[f"{key}_fdr"].fillna(1.0)
        merged[f"{key}_sig"] = merged[f"{key}_sig"].fillna(False)

    blocks = []
    for comp in column_comparisons:
        key = comp["key"]

        top_terms = select_top_n_unique_sig_terms(
            target_key=key,
            enrichment_results=enrichment_results,
            comparisons_for_uniqueness=all_comparisons,
            top_n=top_n,
        )

        if len(top_terms) == 0:
            continue

        block = merged.loc[merged["Term"].isin(top_terms)].copy()
        block["panel_rank"] = block["Term"].map({t: i for i, t in enumerate(top_terms)})
        block["panel"] = key
        block["panel_label"] = comp["topbar_label"]
        block["panel_color"] = comp["topbar_color"]
        block = block.sort_values("panel_rank").copy()
        blocks.append(block)

    if len(blocks) == 0:
        return pd.DataFrame()

    return pd.concat(blocks, axis=0, ignore_index=True)


def pretty_term(term, width=18, mapping_dict=None):
    original = str(term).strip()

    if mapping_dict is not None and original in mapping_dict:
        label = mapping_dict[original]
    else:
        label = original
        label = label.replace("PHENO: ", "")
        label = label.replace("KEGG: ", "KEGG ")
        label = label.replace("REACTOME: ", "Reactome ")
        label = label.replace("BP: ", "GO-BP ")
        label = label.replace("CC: ", "GO-CC ")
        label = label.replace("MF: ", "GO-MF ")
        label = label.replace("_", " ")

    # [10k-scale note] x-labels are rotated vertically (90deg); newline-wrapping
    # would stack wrapped lines side-by-side and collide, so return a single line.
    # scDRS-FM package code is unchanged; this only affects notebook figure labels.
    label = " ".join(str(label).split())
    if len(label) > 55:
        label = label[:52].rstrip() + "..."
    return label


def draw_outlined_text(ax, x, y, text, fontsize, fontweight="normal",
                       color="white", outline_color="black", outline_width=2.0):
    t = ax.text(
        x, y, text,
        ha="center", va="center",
        fontsize=fontsize,
        color=color,
        fontweight=fontweight,
    )
    t.set_path_effects([
        pe.Stroke(linewidth=outline_width, foreground=outline_color),
        pe.Normal()
    ])
    return t


def plot_grouped_multi_heatmap(
    df,
    row_comparisons,
    column_comparisons,
    norm,
    cmap="RdBu_r",
    mapping_dict=None,
    gene_counts: Mapping[str, int] | None = None,
):
    if df.empty:
        raise ValueError("No pathways available after filtering.")

    values = np.vstack([
        df[f"{comp['key']}_or"].to_numpy(dtype=float)
        for comp in row_comparisons
    ])

    sigs = np.vstack([
        df[f"{comp['key']}_sig"].astype(bool).to_numpy()
        for comp in row_comparisons
    ])

    xlabels = [pretty_term(x, width=TERM_WRAP_WIDTH, mapping_dict=mapping_dict) for x in df["Term"]]

    ncols = len(df)
    nrows = len(row_comparisons)

    fig_w = max(18, 1.15 * ncols + 6)  # [10k-scale note] wider per-column spacing so pathway labels do not overlap (package code unchanged)
    fig_h = max(5, 0.75 * nrows + 3)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(values, aspect="equal", cmap=cmap, norm=norm)

    ax.set_xticks(np.arange(ncols))
    ax.set_xticklabels(xlabels, rotation=XTICK_ROTATION, ha="center", fontsize=XTICK_FONTSIZE)

    ax.set_yticks(np.arange(nrows))
    gene_counts = {} if gene_counts is None else gene_counts
    row_labels = []
    for comp in row_comparisons:
        title = comp["title"]
        if comp["key"] in gene_counts:
            title = f"{title} ({int(gene_counts[comp['key']]):,} genes)"
        row_labels.append(title)
    ax.set_yticklabels(row_labels, fontsize=YTICK_FONTSIZE)

    ax.set_xticks(np.arange(-0.5, ncols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, nrows, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.6)
    ax.tick_params(which="minor", bottom=False, left=False)

    for sp in ax.spines.values():
        sp.set_visible(False)

    # Numerical odds-ratio annotations only; no significance stars.
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if sigs[i, j]:
                txt = f"{values[i, j]:.1f}"
                draw_outlined_text(
                    ax,
                    x=j,
                    y=i,
                    text=txt,
                    fontsize=ANNOT_FONTSIZE,
                    fontweight=ANNOT_FONTWEIGHT,
                    color="white",
                    outline_color=ANNOT_OUTLINE_COLOR,
                    outline_width=ANNOT_OUTLINE_WIDTH,
                )

    ax.set_xlim(-0.5, ncols - 0.5)
    ax.set_ylim(nrows - 0.5, -1.35)

    bar_y = -0.96
    label_y = -1.65

    start_idx = 0
    for idx, comp in enumerate(column_comparisons):
        block_size = int((df["panel"] == comp["key"]).sum())
        if block_size == 0:
            continue

        x0 = start_idx - 0.5
        x1 = start_idx + block_size - 0.5

        ax.plot(
            [x0, x1],
            [bar_y, bar_y],
            color=comp["topbar_color"],
            linewidth=6,
            solid_capstyle="butt",
            clip_on=False
        )

        ax.text(
            start_idx + (block_size - 1) / 2,
            label_y - 0.1,
            comp["topbar_label"],
            ha="center",
            va="center",
            fontsize=TOPBAR_LABEL_FONTSIZE,
            fontweight=TOPBAR_LABEL_FONTWEIGHT,
            color=comp["topbar_color"],
            clip_on=False
        )

        start_idx += block_size

        if idx < (len(column_comparisons) - 1):
            ax.vlines(
                x=start_idx - 0.5,
                ymin=-0.5,
                ymax=nrows - 0.5,
                color="black",
                linewidth=1.5
            )

    cbar = fig.colorbar(im, ax=ax, orientation="vertical", pad=0.02, fraction=0.04)
    cbar.set_label("Odds ratio", fontsize=CBAR_LABEL_FONTSIZE)
    cbar.ax.tick_params(labelsize=CBAR_TICK_FONTSIZE)

    plt.tight_layout()

    if SAVEPATH is not None:
        plt.savefig(SAVEPATH, dpi=DPI, bbox_inches="tight")

    plt.show()

# ============================================================
# Background genes for ORA
# ============================================================
background_gene_sets = []
for comp in COMPARISONS:
    key = comp["key"]
    ranked = prep_ranked_de_table(deg_results[key]["all_ranked_genes"])
    background_gene_sets.append(set(ranked["names"]))

BACKGROUND_GENES = sorted(set().union(*background_gene_sets))

# ============================================================
# Positive DE genes + enrichment
# ============================================================
pos_deg_genes = {}
enrichment_results = {}

for comp in COMPARISONS:
    key = comp["key"]
    de_table = deg_results[key]["all_ranked_genes"].copy()

    pos, _ = split_pos_neg_deg_genes(de_table, padj_cutoff=DE_PADJ_CUTOFF)
    pos_deg_genes[key] = pos

    enrichment_results[key] = run_enrichr_dict(
        pos,
        ALL_GENESETS,
        background_genes=BACKGROUND_GENES,
        enrich_fdr_cutoff=ENRICH_FDR_CUTOFF,
        min_recall=MIN_RECALL,
        min_intersect=MIN_INTERSECT,
    )

# Number of positive DE genes used as the enrichment query for each row.
# These counts are displayed in brackets in the final heatmap labels.
positive_de_gene_counts = {
    comp["key"]: len(pos_deg_genes[comp["key"]])
    for comp in COMPARISONS
}

# ============================================================
# Build grouped plot table
# - rows are all 5 comparisons
# - columns now also include both background comparison groups
# ============================================================
plot_df = build_grouped_topn_table_multi(
    enrichment_results,
    all_comparisons=COMPARISONS,
    column_comparisons=COLUMN_COMPARISONS,
    top_n=TOP_N_PER_POPULATION,
)

if plot_df.empty:
    raise ValueError("No uniquely significant enriched pathways found for the selected column groups.")

display(plot_df[["Term", "panel"]].head(50))

# ============================================================
# Plot
# ============================================================
sns.set_style("white")
norm = mcolors.TwoSlopeNorm(vmin=OR_VMIN, vcenter=OR_VCENTER, vmax=OR_VMAX)

plot_grouped_multi_heatmap(
    plot_df,
    row_comparisons=COMPARISONS,
    column_comparisons=COLUMN_COMPARISONS,
    norm=norm,
    cmap="RdBu_r",
    mapping_dict=mapping_dict,
    gene_counts=positive_de_gene_counts,
)

[10k-scale note] pathway-heatmap comparisons pruned to computed keys: ['signal1_vs_rest_microglia_union', 'microglia_regions_vs_rest_microglia_union', 'human_mtg_it_regions_vs_rest_IT_union']


,Term,panel
0,BP: Post-translational Protein Modification (G...,signal1_vs_rest_microglia_union
1,BP: Regulation of Protein Localization to Nucl...,signal1_vs_rest_microglia_union
2,REACTOME: HCMV Late Events,signal1_vs_rest_microglia_union
3,REACTOME: Classical Antibody-Mediated Compleme...,microglia_regions_vs_rest_microglia_union
4,BP: Regulation of Trans-Synaptic Signaling (GO...,microglia_regions_vs_rest_microglia_union
5,BP: Neuron Migration (GO:0001764),microglia_regions_vs_rest_microglia_union
6,BP: Oxidative Phosphorylation (GO:0006119),human_mtg_it_regions_vs_rest_IT_union
7,BP: Proton Motive Force-Driven ATP Synthesis (...,human_mtg_it_regions_vs_rest_IT_union
8,BP: Proton Motive Force-Driven Mitochondrial A...,human_mtg_it_regions_vs_rest_IT_union


/tmp/ipykernel_3139/3358770896.py:539: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


In [27]:
list(plot_df['Term'].unique())

['BP: Post-translational Protein Modification (GO:0043687)',
 'BP: Regulation of Protein Localization to Nucleus (GO:1900180)',
 'REACTOME: HCMV Late Events',
 'REACTOME: Classical Antibody-Mediated Complement Activation',
 'BP: Regulation of Trans-Synaptic Signaling (GO:0099177)',
 'BP: Neuron Migration (GO:0001764)',
 'BP: Oxidative Phosphorylation (GO:0006119)',
 'BP: Proton Motive Force-Driven ATP Synthesis (GO:0015986)',
 'BP: Proton Motive Force-Driven Mitochondrial ATP Synthesis (GO:0042776)']